In [2]:
import sys
import subprocess
import importlib.util

REQUIRED_PACKAGES = [
    "numpy",
    "pandas",
]

missing = [
    pkg for pkg in REQUIRED_PACKAGES
    if importlib.util.find_spec(pkg) is None
]

if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("Core packages already installed.")
    
from __future__ import annotations

import math
import os
import re
import warnings
from contextlib import closing
from dataclasses import dataclass
from datetime import date, timedelta
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None

try:
    display
except NameError:
    def display(obj):
        print(obj)

warnings.filterwarnings("ignore", category=RuntimeWarning)
try:
    warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)
except Exception:
    pass

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)


Installing missing packages: ['numpy', 'pandas']
  Using cached numpy-2.5.0-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached pandas-3.0.3-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
Using cached numpy-2.5.0-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.6 MB)
Using cached pandas-3.0.3-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (10.9 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]


In [3]:
# 安装依赖
!pip install -U tqsdk pandas numpy statsmodels arch matplotlib openpyxl

  Using cached tqsdk-3.10.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached arch-8.0.0-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (13 kB)
  Using cached matplotlib-3.11.0-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (80 kB)
  Using cached websockets-16.0-cp313-cp313-manylinux1_x86_64.manylinux_2_28_x86_64.manylinux_2_5_x86_64.whl.metadata (6.8 kB)
  Using cached simplejson-4.1.1-cp313-cp313-manylinux1_x86_64.manylinux_2_28_x86_64.manylinux_2_5_x86_64.whl.metadata (3.8 kB)
  Using cached shinny_structlog-0.0.4-py3-none-any.whl.metadata (23 kB)
  Using cached sgqlc-18-py3-none-any.whl.metadata (19 kB)
  Using cached tqsdk_ctpse-1.1.0-py3-none-manylinux1_x86_64.whl.metadata (248 bytes)
  Using cached tqsdk_sm-1.0.5-py3-none-manylinux1_x86_64.whl.metadata (242 bytes)
  Using cached contourpy-1.3.3-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl

In [4]:
PROJECT_DIR = Path("main_continuous_daily_trend_momentum_reversal_research")
RAW_MAIN_DIR = PROJECT_DIR / "raw" / "tq_main_continuous_daily"
PROCESSED_DIR = PROJECT_DIR / "processed"
RESULT_DIR = PROJECT_DIR / "results"
FIG_DIR = PROJECT_DIR / "figures"

for p in [RAW_MAIN_DIR, PROCESSED_DIR, RESULT_DIR, FIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# 运行开关
RUN_TQ_DOWNLOAD = True
AUTO_DOWNLOAD_IF_MISSING = True
OVERWRITE_RAW_CSV = True
RUN_PLOTS = False

# 日线参数
DAILY_KLINE_DATA_LENGTH = 8000
DAILY_KLINE_DUR_SEC = 24 * 60 * 60

# 研究区间
START_DATE = date(1990, 1, 1)
END_DATE = date.today()
LOCAL_TIMEZONE = "Asia/Shanghai"

# 品种起始过滤日期
PRODUCT_START_DATE = {
    "M": date(2000, 1, 1),
    "SP": date(2018, 11, 27),
    "FG": date(2012, 12, 3),
    "SA": date(2019, 12, 6),

    "AG": date(2012, 5, 10),
    "V": date(2009, 5, 25),
    "LH": date(2021, 1, 8),
    "EB": date(2019, 9, 26),
    "MA": date(2011, 10, 28),
    "CF": date(2004, 6, 1),
    "SR": date(2006, 1, 6),
}

MIN_OHLCV_VOLUME = 0
MIN_SIGMA = 1e-6
VOL_BASE_LOOKBACK = 20
EWMA_LAMBDA = 0.94

H_RET_LIST = [1, 2, 3, 5, 10, 20, 40, 60]
H_TREND_LIST = [10, 20, 40, 60]
J_LIST = [2, 3, 5, 10, 20, 40, 60]
K_LIST = [1, 2, 3, 5, 10, 20]
L_LIST = [0, 2, 3, 5, 10, 20]

TREND_T_THRESHOLD = 2.0
TREND_FIT_THRESHOLD = 0.20
TREND_OVEREXT_Q = 0.90
ATR_STOP_LOOKBACK = 20
ATR_STOP_MULT = 2.5

MOM_Z_THRESHOLD = 1.0
MCS_THRESHOLD = 0.58
SHOCK_Z_THRESHOLD = 2.0
VOLUME_CONFIRM_Z = 0.25

FAST_REV_Z = 2.0
REV_SIGNAL_MIN_ABS = 1e-12

MIN_MODEL_N = 60
MIN_CONFIRMED_N = 120
MIN_T_CAND = 1.0
MIN_T_CONF = 2.0
MIN_HIT_RATE = 0.50
BOOTSTRAP_BLOCK = 5
BOOTSTRAP_N = 400
RANDOM_SEED = 20260630

rng = np.random.default_rng(RANDOM_SEED)


In [5]:
@dataclass(frozen=True)
class ProductSpec:
    commodity: str
    name: str
    main_symbol: str


PRODUCTS = {
    "M": ProductSpec("M", "豆粕", "KQ.m@DCE.m"),
    "SP": ProductSpec("SP", "纸浆", "KQ.m@SHFE.sp"),
    "FG": ProductSpec("FG", "玻璃", "KQ.m@CZCE.FG"),
    "SA": ProductSpec("SA", "纯碱", "KQ.m@CZCE.SA"),

    "AG": ProductSpec("AG", "沪银", "KQ.m@SHFE.ag"),
    "V": ProductSpec("V", "PVC", "KQ.m@DCE.v"),
    "LH": ProductSpec("LH", "生猪", "KQ.m@DCE.lh"),
    "EB": ProductSpec("EB", "苯乙烯", "KQ.m@DCE.eb"),
    "MA": ProductSpec("MA", "甲醇", "KQ.m@CZCE.MA"),
    "CF": ProductSpec("CF", "棉花", "KQ.m@CZCE.CF"),
    "SR": ProductSpec("SR", "白糖", "KQ.m@CZCE.SR"),
}

# 天勤账号
TQ_USERNAME = os.getenv("TQ_USERNAME", "")
TQ_PASSWORD = os.getenv("TQ_PASSWORD", "")

print("研究品种：")
for k, v in PRODUCTS.items():
    print(f"  {k}: {v.main_symbol}, name={v.name}")

print("\n是否在 Cell 04 主动下载：", RUN_TQ_DOWNLOAD)
print("缺失数据时是否自动下载：", AUTO_DOWNLOAD_IF_MISSING)
print("当前工作目录：", Path.cwd().resolve())
print("主力连续合约 CSV 目录：", RAW_MAIN_DIR.resolve())


研究品种：
  M: KQ.m@DCE.m, name=豆粕
  SP: KQ.m@SHFE.sp, name=纸浆
  FG: KQ.m@CZCE.FG, name=玻璃
  SA: KQ.m@CZCE.SA, name=纯碱
  AG: KQ.m@SHFE.ag, name=沪银
  V: KQ.m@DCE.v, name=PVC
  LH: KQ.m@DCE.lh, name=生猪
  EB: KQ.m@DCE.eb, name=苯乙烯
  MA: KQ.m@CZCE.MA, name=甲醇
  CF: KQ.m@CZCE.CF, name=棉花
  SR: KQ.m@CZCE.SR, name=白糖

是否在 Cell 04 主动下载： True
缺失数据时是否自动下载： True
当前工作目录： /home/zilinm2
主力连续合约 CSV 目录： /home/zilinm2/main_continuous_daily_trend_momentum_reversal_research/raw/tq_main_continuous_daily


In [22]:
def safe_filename(symbol: str) -> str:
    return re.sub(r"[^A-Za-z0-9_]+", "_", symbol)


def main_csv_path(commodity: str, symbol: str) -> Path:
    return RAW_MAIN_DIR / f"{commodity}_{safe_filename(symbol)}_daily.csv"


def has_tq_credentials() -> bool:
    return bool(str(TQ_USERNAME).strip()) and bool(str(TQ_PASSWORD).strip())


def create_tq_api():
    from tqsdk import TqApi, TqAuth

    if not has_tq_credentials():
        raise RuntimeError(
            "缺少天勤账号。请设置环境变量 TQ_USERNAME / TQ_PASSWORD，"
            "或在 Cell 03 中填写 TQ_USERNAME / TQ_PASSWORD。"
        )

    return TqApi(auth=TqAuth(TQ_USERNAME, TQ_PASSWORD))


def parse_numeric_datetime_series(x: pd.Series) -> pd.Series:
    """
    解析 TqSdk K 线 datetime。
    关键点：get_kline_serial 前部可能有 datetime=0 的占位行，不能参与时间单位判断。
    """
    x_num = pd.to_numeric(x, errors="coerce")
    positive = x_num[x_num > 0].dropna()

    out = pd.Series(pd.NaT, index=x.index, dtype="datetime64[ns]")

    if positive.empty:
        return out

    med = positive.median()
    valid = x_num > 0

    if 19000101 <= med <= 21001231:
        as_text = x_num.loc[valid].round().astype("Int64").astype(str).str.zfill(8)
        out.loc[valid] = pd.to_datetime(as_text, format="%Y%m%d", errors="coerce")
        return out

    if med > 1e17:
        unit = "ns"
    elif med > 1e14:
        unit = "us"
    elif med > 1e11:
        unit = "ms"
    else:
        unit = "s"

    dt = pd.to_datetime(x_num.loc[valid], unit=unit, errors="coerce", utc=True)
    out.loc[valid] = dt.dt.tz_convert(LOCAL_TIMEZONE).dt.tz_localize(None)

    return out


def parse_datetime_series(s: pd.Series) -> pd.Series:
    if pd.api.types.is_numeric_dtype(s):
        return parse_numeric_datetime_series(s)

    text = s.astype(str).str.strip()
    numeric = pd.to_numeric(text, errors="coerce")

    if numeric.notna().mean() >= 0.80:
        return parse_numeric_datetime_series(numeric)

    return pd.to_datetime(text, errors="coerce")


def normalize_kline_serial_to_daily(
    kline_df: pd.DataFrame,
    commodity: str,
    symbol: str,
    csv_path: Path,
) -> pd.DataFrame:
    """
    将 TqSdk 主力连续合约日线 K 线标准化为研究用 OHLCV。
    """
    if kline_df is None or len(kline_df) == 0:
        return pd.DataFrame()

    raw = kline_df.copy()
    raw.columns = [str(c).strip() for c in raw.columns]
    lower_map = {c.lower(): c for c in raw.columns}

    required = ["datetime", "open", "high", "low", "close", "volume"]
    missing = [c for c in required if c not in lower_map]
    if missing:
        raise ValueError(f"日线 K 线缺少字段：{missing}；当前字段：{list(raw.columns)}")

    df = pd.DataFrame()
    df["datetime"] = parse_datetime_series(raw[lower_map["datetime"]])
    df["date"] = df["datetime"].dt.normalize()

    for c in ["open", "high", "low", "close", "volume"]:
        df[c] = pd.to_numeric(raw[lower_map[c]], errors="coerce")

    df["commodity"] = commodity
    df["commodity_name"] = PRODUCTS[commodity].name
    df["main_symbol"] = symbol
    df["data_source"] = "TQ_MAIN_CONTINUOUS_KLINE_DAILY"
    df["csv_path"] = str(csv_path)

    start_ts = pd.Timestamp(max(PRODUCT_START_DATE.get(commodity, START_DATE), START_DATE))
    end_ts = pd.Timestamp(END_DATE)

    valid = (
        df["date"].notna()
        & (df["date"] >= start_ts)
        & (df["date"] <= end_ts)
        & (df[["open", "high", "low", "close"]] > 0).all(axis=1)
        & (df["volume"].fillna(-1) >= MIN_OHLCV_VOLUME)
        & (df["high"] >= df[["open", "close", "low"]].max(axis=1))
        & (df["low"] <= df[["open", "close", "high"]].min(axis=1))
    )

    df = df.loc[valid].copy()

    if df.empty:
        return pd.DataFrame()

    df = df.sort_values(["date", "datetime"]).drop_duplicates(["date"], keep="last")

    keep_cols = [
        "date", "datetime", "commodity", "commodity_name", "main_symbol", "data_source",
        "open", "high", "low", "close", "volume", "csv_path",
    ]

    return df[keep_cols].reset_index(drop=True)


def tq_wait_update_safe(api, deadline: int = 1) -> None:
    try:
        api.wait_update(deadline=deadline)
    except TypeError:
        api.wait_update()


def fetch_one_daily_kline_csv(
    api,
    commodity: str,
    symbol: str,
    csv_path: Path,
    overwrite: bool = False,
    data_length: int = DAILY_KLINE_DATA_LENGTH,
) -> Tuple[bool, str, int]:
    """
    直接抓取 TqSdk 主力连续合约日线 K 线并保存 CSV。
    """
    if csv_path.exists() and not overwrite:
        existing = pd.read_csv(csv_path)
        return True, "existing", int(len(existing))

    csv_path.parent.mkdir(parents=True, exist_ok=True)

    try:
        klines = api.get_kline_serial(
            symbol,
            DAILY_KLINE_DUR_SEC,
            data_length=data_length,
        )

        if hasattr(api, "is_serial_ready"):
            wait_count = 0
            while not api.is_serial_ready(klines) and wait_count < 200:
                tq_wait_update_safe(api, deadline=1)
                wait_count += 1
        else:
            for _ in range(20):
                tq_wait_update_safe(api, deadline=1)

        daily_panel = normalize_kline_serial_to_daily(
            klines.copy(),
            commodity,
            symbol,
            csv_path,
        )

        if daily_panel.empty:
            return False, "日线 K 线为空或清洗后无有效 OHLCV", 0

        daily_panel.to_csv(csv_path, index=False)
        return True, "", int(len(daily_panel))

    except Exception as e:
        return False, str(e), 0


def download_main_continuous_for_products(
    products: Dict[str, ProductSpec] = PRODUCTS,
    overwrite: bool = OVERWRITE_RAW_CSV,
) -> pd.DataFrame:
    rows = []

    with closing(create_tq_api()) as api:
        for commodity, spec in products.items():
            csv_path = main_csv_path(commodity, spec.main_symbol)

            ok, err, rows_count = fetch_one_daily_kline_csv(
                api=api,
                commodity=commodity,
                symbol=spec.main_symbol,
                csv_path=csv_path,
                overwrite=overwrite,
            )

            rows.append({
                "commodity": commodity,
                "symbol": spec.main_symbol,
                "csv_path": str(csv_path),
                "ok": ok,
                "rows": rows_count,
                "error": err,
            })

            print(f"[主连日线] {commodity}: ok={ok}, rows={rows_count}, csv={csv_path}")

    report = pd.DataFrame(rows)
    report.to_csv(RESULT_DIR / "tq_main_continuous_daily_download_report.csv", index=False)
    return report


if RUN_TQ_DOWNLOAD:
    if has_tq_credentials():
        main_download_report = download_main_continuous_for_products()
        display(main_download_report)
    else:
        print("RUN_TQ_DOWNLOAD=True，但尚未填写天勤账号。请在 Cell 03 填写账号后重跑 Cell 03-04。")
else:
    print("RUN_TQ_DOWNLOAD=False。若本地 CSV 缺失且已填写天勤账号，后续 Cell 会自动抓取日线 K 线。")

2026-06-30 09:45:56 -     INFO - 通知 : 与 wss://free-api.shinnytech.com/t/nfmd/front/mobile 的网络连接已建立
[主连日线] M: ok=True, rows=2545, csv=main_continuous_daily_trend_momentum_reversal_research/raw/tq_main_continuous_daily/M_KQ_m_DCE_m_daily.csv
[主连日线] SP: ok=True, rows=1839, csv=main_continuous_daily_trend_momentum_reversal_research/raw/tq_main_continuous_daily/SP_KQ_m_SHFE_sp_daily.csv
[主连日线] FG: ok=True, rows=2545, csv=main_continuous_daily_trend_momentum_reversal_research/raw/tq_main_continuous_daily/FG_KQ_m_CZCE_FG_daily.csv
[主连日线] SA: ok=True, rows=1589, csv=main_continuous_daily_trend_momentum_reversal_research/raw/tq_main_continuous_daily/SA_KQ_m_CZCE_SA_daily.csv
[主连日线] AG: ok=True, rows=2545, csv=main_continuous_daily_trend_momentum_reversal_research/raw/tq_main_continuous_daily/AG_KQ_m_SHFE_ag_daily.csv
[主连日线] V: ok=True, rows=2545, csv=main_continuous_daily_trend_momentum_reversal_research/raw/tq_main_continuous_daily/V_KQ_m_DCE_v_daily.csv
[主连日线] LH: ok=True, rows=1324, csv=main

,commodity,symbol,csv_path,ok,rows,error
0,M,KQ.m@DCE.m,main_continuous_daily_trend_momentum_reversal_...,True,2545,
1,SP,KQ.m@SHFE.sp,main_continuous_daily_trend_momentum_reversal_...,True,1839,
2,FG,KQ.m@CZCE.FG,main_continuous_daily_trend_momentum_reversal_...,True,2545,
3,SA,KQ.m@CZCE.SA,main_continuous_daily_trend_momentum_reversal_...,True,1589,
4,AG,KQ.m@SHFE.ag,main_continuous_daily_trend_momentum_reversal_...,True,2545,
5,V,KQ.m@DCE.v,main_continuous_daily_trend_momentum_reversal_...,True,2545,
6,LH,KQ.m@DCE.lh,main_continuous_daily_trend_momentum_reversal_...,True,1324,
7,EB,KQ.m@DCE.eb,main_continuous_daily_trend_momentum_reversal_...,True,1635,
8,MA,KQ.m@CZCE.MA,main_continuous_daily_trend_momentum_reversal_...,True,2545,
9,CF,KQ.m@CZCE.CF,main_continuous_daily_trend_momentum_reversal_...,True,2545,


In [23]:
def normalize_tq_daily_csv(csv_path: Path, commodity: str, symbol: str) -> pd.DataFrame:
    raw = pd.read_csv(csv_path)
    raw.columns = [str(c).strip() for c in raw.columns]
    lower_map = {c.lower(): c for c in raw.columns}

    dt_col = None
    for cand in ["datetime", "date", "time", "trading_day"]:
        if cand in lower_map:
            dt_col = lower_map[cand]
            break
    if dt_col is None:
        raise ValueError(f"{csv_path} 没有 datetime/date/time/trading_day 字段")

    col_map = {}
    for std in ["open", "high", "low", "close", "volume"]:
        if std in lower_map:
            col_map[lower_map[std]] = std

    missing = sorted(set(["open", "high", "low", "close", "volume"]) - set(col_map.values()))
    if missing:
        raise ValueError(f"{csv_path} 缺少 OHLCV 字段：{missing}")

    df = raw[[dt_col] + list(col_map.keys())].rename(columns={dt_col: "datetime", **col_map})
    df["datetime"] = parse_datetime_series(df["datetime"])
    df["date"] = df["datetime"].dt.normalize()

    for c in ["open", "high", "low", "close", "volume"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df["commodity"] = commodity
    df["commodity_name"] = PRODUCTS[commodity].name
    df["main_symbol"] = symbol

    if "data_source" in lower_map:
        source_val = (
            str(raw[lower_map["data_source"]].dropna().iloc[0])
            if raw[lower_map["data_source"]].notna().any()
            else "TQ_MAIN_CONTINUOUS_KLINE_DAILY"
        )
    else:
        source_val = "TQ_MAIN_CONTINUOUS_KLINE_DAILY"

    df["data_source"] = source_val
    df["csv_path"] = str(csv_path)

    start_ts = pd.Timestamp(max(PRODUCT_START_DATE.get(commodity, START_DATE), START_DATE))
    end_ts = pd.Timestamp(END_DATE)

    valid = (
        df["date"].notna()
        & (df["date"] >= start_ts)
        & (df["date"] <= end_ts)
        & (df[["open", "high", "low", "close"]] > 0).all(axis=1)
        & (df["volume"].fillna(-1) >= MIN_OHLCV_VOLUME)
        & (df["high"] >= df[["open", "close", "low"]].max(axis=1))
        & (df["low"] <= df[["open", "close", "high"]].min(axis=1))
    )

    df = df.loc[valid].copy()

    keep_cols = [
        "date", "datetime", "commodity", "commodity_name", "main_symbol",
        "data_source", "open", "high", "low", "close", "volume", "csv_path",
    ]

    df = df[keep_cols]
    df = df.sort_values(["commodity", "date", "datetime"])
    df = df.drop_duplicates(["commodity", "date"], keep="last")

    return df.reset_index(drop=True)


def list_main_csvs(commodity: str) -> List[Path]:
    spec = PRODUCTS[commodity]
    expected = main_csv_path(commodity, spec.main_symbol)
    if expected.exists():
        return [expected]
    return sorted(
        RAW_MAIN_DIR.glob(f"{commodity}_*_daily.csv"),
        key=lambda p: (p.stat().st_mtime, p.name),
    )

In [24]:
def missing_required_commodities(df: pd.DataFrame) -> List[str]:
    present = set(df["commodity"].dropna().unique()) if "commodity" in df.columns and not df.empty else set()
    return [commodity for commodity in PRODUCTS if commodity not in present]


def load_main_continuous_panel() -> pd.DataFrame:
    frames = []
    errors = []

    for commodity, spec in PRODUCTS.items():
        paths = list_main_csvs(commodity)
        if not paths:
            print(f"[WARN] 未发现 {commodity} 的主力连续合约日线 CSV")
            continue
        path = paths[-1]
        try:
            df = normalize_tq_daily_csv(path, commodity, symbol=spec.main_symbol)
            if len(df) > 0:
                frames.append(df)
        except Exception as e:
            errors.append({"commodity": commodity, "csv_path": str(path), "error": str(e)})

    if errors:
        pd.DataFrame(errors).to_csv(RESULT_DIR / "load_main_continuous_errors.csv", index=False)
        print(f"[WARN] 主力连续合约日线 CSV 读取错误数量：{len(errors)}")

    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


def load_research_panel() -> pd.DataFrame:
    main_panel = load_main_continuous_panel()
    missing = missing_required_commodities(main_panel)

    if missing and AUTO_DOWNLOAD_IF_MISSING:
        if has_tq_credentials():
            print(f"[AUTO] 主力连续合约日线 CSV 不完整，开始补齐：{missing}")
            report = download_main_continuous_for_products({c: PRODUCTS[c] for c in missing})
            display(report)
            main_panel = load_main_continuous_panel()
            missing = missing_required_commodities(main_panel)
        else:
            print(
                f"[AUTO] 主力连续合约日线 CSV 不完整，缺失 {missing}，且未填写天勤账号。"
                "请在 Cell 03 填写 TQ_USERNAME / TQ_PASSWORD 后，重新运行 Cell 03-06。"
            )

    if not missing:
        print(f"[OK] 已载入主力连续合约数据：{len(main_panel):,} 行，{main_panel['commodity'].nunique()} 个标的")
        return main_panel

    raise RuntimeError(
        f"未能载入完整 {len(PRODUCTS)} 个主力连续合约日线数据，缺失：{missing}。请处理以下任一项：\n"
        "1. 在 Cell 03 填写 TQ_USERNAME / TQ_PASSWORD，并重新运行 Cell 03-06；或\n"
        "2. 设置 RUN_TQ_DOWNLOAD=True，并从 Cell 02 开始重新运行；或\n"
        f"3. 将 {len(PRODUCTS)} 个日线 CSV 放入 {RAW_MAIN_DIR}/。"
    )


panel_raw = load_research_panel()
display(panel_raw.head())
display(panel_raw.groupby(["commodity", "main_symbol"]).agg(
    rows=("date", "size"),
    start=("date", "min"),
    end=("date", "max"),
))

[OK] 已载入主力连续合约数据：24,202 行，11 个标的


,date,datetime,commodity,commodity_name,main_symbol,data_source,open,high,low,close,volume,csv_path
0,2016-01-05,2016-01-05,M,豆粕,KQ.m@DCE.m,TQ_MAIN_CONTINUOUS_KLINE_DAILY,2327.0,2347.0,2325.0,2343.0,569527.0,main_continuous_daily_trend_momentum_reversal_...
1,2016-01-06,2016-01-06,M,豆粕,KQ.m@DCE.m,TQ_MAIN_CONTINUOUS_KLINE_DAILY,2345.0,2352.0,2327.0,2347.0,683608.0,main_continuous_daily_trend_momentum_reversal_...
2,2016-01-07,2016-01-07,M,豆粕,KQ.m@DCE.m,TQ_MAIN_CONTINUOUS_KLINE_DAILY,2341.0,2386.0,2339.0,2376.0,987863.0,main_continuous_daily_trend_momentum_reversal_...
3,2016-01-08,2016-01-08,M,豆粕,KQ.m@DCE.m,TQ_MAIN_CONTINUOUS_KLINE_DAILY,2375.0,2407.0,2363.0,2404.0,989187.0,main_continuous_daily_trend_momentum_reversal_...
4,2016-01-11,2016-01-11,M,豆粕,KQ.m@DCE.m,TQ_MAIN_CONTINUOUS_KLINE_DAILY,2407.0,2413.0,2377.0,2383.0,745035.0,main_continuous_daily_trend_momentum_reversal_...


,,rows,start,end
commodity,main_symbol,,,
AG,KQ.m@SHFE.ag,2545,2016-01-05,2026-06-30
CF,KQ.m@CZCE.CF,2545,2016-01-05,2026-06-30
EB,KQ.m@DCE.eb,1635,2019-09-26,2026-06-30
FG,KQ.m@CZCE.FG,2545,2016-01-05,2026-06-30
LH,KQ.m@DCE.lh,1324,2021-01-08,2026-06-30
M,KQ.m@DCE.m,2545,2016-01-05,2026-06-30
MA,KQ.m@CZCE.MA,2545,2016-01-05,2026-06-30
SA,KQ.m@CZCE.SA,1589,2019-12-06,2026-06-30
SP,KQ.m@SHFE.sp,1839,2018-11-27,2026-06-30


In [25]:
def clean_ohlcv_panel(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out = out.sort_values(["commodity", "date"]).reset_index(drop=True)

    for c in ["open", "high", "low", "close", "volume"]:
        out[c] = pd.to_numeric(out[c], errors="coerce")

    bad = (
        out[["open", "high", "low", "close"]].isna().any(axis=1)
        | (out[["open", "high", "low", "close"]] <= 0).any(axis=1)
        | (out["volume"].fillna(-1) < MIN_OHLCV_VOLUME)
        | (out["high"] < out[["open", "close", "low"]].max(axis=1))
        | (out["low"] > out[["open", "close", "high"]].min(axis=1))
    )

    out["ohlcv_bad"] = bad.astype(float)
    out = out.loc[~bad].copy()

    out["series_age"] = out.groupby("commodity").cumcount() + 1
    out["series_len"] = out.groupby("commodity")["date"].transform("size")
    out["series_remain"] = out["series_len"] - out["series_age"]
    out["series_start"] = out.groupby("commodity")["date"].transform("min")
    out["series_end"] = out.groupby("commodity")["date"].transform("max")
    out["data_reliability"] = "MAIN_CONTINUOUS_DIAGNOSTIC"

    return out.sort_values(["commodity", "date"]).reset_index(drop=True)


panel = clean_ohlcv_panel(panel_raw)
missing_after_cleaning = missing_required_commodities(panel)
if missing_after_cleaning:
    raise RuntimeError(f"OHLCV 清洗后缺失品种：{missing_after_cleaning}。请检查对应主力连续合约日线 CSV。")

display(panel.head())
display(panel.groupby(["commodity", "data_reliability"]).agg(
    rows=("date", "size"),
    start=("date", "min"),
    end=("date", "max"),
))

,date,datetime,commodity,commodity_name,main_symbol,data_source,open,high,low,close,volume,csv_path,ohlcv_bad,series_age,series_len,series_remain,series_start,series_end,data_reliability
0,2016-01-05,2016-01-05,AG,沪银,KQ.m@SHFE.ag,TQ_MAIN_CONTINUOUS_KLINE_DAILY,3306.0,3330.0,3294.0,3313.0,287111.0,main_continuous_daily_trend_momentum_reversal_...,0.0,1,2545,2544,2016-01-05,2026-06-30,MAIN_CONTINUOUS_DIAGNOSTIC
1,2016-01-06,2016-01-06,AG,沪银,KQ.m@SHFE.ag,TQ_MAIN_CONTINUOUS_KLINE_DAILY,3314.0,3321.0,3296.0,3314.0,204526.0,main_continuous_daily_trend_momentum_reversal_...,0.0,2,2545,2543,2016-01-05,2026-06-30,MAIN_CONTINUOUS_DIAGNOSTIC
2,2016-01-07,2016-01-07,AG,沪银,KQ.m@SHFE.ag,TQ_MAIN_CONTINUOUS_KLINE_DAILY,3322.0,3385.0,3308.0,3357.0,488186.0,main_continuous_daily_trend_momentum_reversal_...,0.0,3,2545,2542,2016-01-05,2026-06-30,MAIN_CONTINUOUS_DIAGNOSTIC
3,2016-01-08,2016-01-08,AG,沪银,KQ.m@SHFE.ag,TQ_MAIN_CONTINUOUS_KLINE_DAILY,3367.0,3390.0,3345.0,3356.0,379737.0,main_continuous_daily_trend_momentum_reversal_...,0.0,4,2545,2541,2016-01-05,2026-06-30,MAIN_CONTINUOUS_DIAGNOSTIC
4,2016-01-11,2016-01-11,AG,沪银,KQ.m@SHFE.ag,TQ_MAIN_CONTINUOUS_KLINE_DAILY,3345.0,3362.0,3322.0,3349.0,283036.0,main_continuous_daily_trend_momentum_reversal_...,0.0,5,2545,2540,2016-01-05,2026-06-30,MAIN_CONTINUOUS_DIAGNOSTIC


,,rows,start,end
commodity,data_reliability,,,
AG,MAIN_CONTINUOUS_DIAGNOSTIC,2545,2016-01-05,2026-06-30
CF,MAIN_CONTINUOUS_DIAGNOSTIC,2545,2016-01-05,2026-06-30
EB,MAIN_CONTINUOUS_DIAGNOSTIC,1635,2019-09-26,2026-06-30
FG,MAIN_CONTINUOUS_DIAGNOSTIC,2545,2016-01-05,2026-06-30
LH,MAIN_CONTINUOUS_DIAGNOSTIC,1324,2021-01-08,2026-06-30
M,MAIN_CONTINUOUS_DIAGNOSTIC,2545,2016-01-05,2026-06-30
MA,MAIN_CONTINUOUS_DIAGNOSTIC,2545,2016-01-05,2026-06-30
SA,MAIN_CONTINUOUS_DIAGNOSTIC,1589,2019-12-06,2026-06-30
SP,MAIN_CONTINUOUS_DIAGNOSTIC,1839,2018-11-27,2026-06-30


In [26]:
def nw_tstat(x: Sequence[float], lags: Optional[int] = None) -> Tuple[float, float]:
    arr = pd.Series(x).dropna().astype(float).to_numpy()
    n = len(arr)
    if n < 3:
        return np.nan, np.nan

    mean = float(np.mean(arr))
    demeaned = arr - mean

    if lags is None:
        lags = int(np.floor(4 * (n / 100) ** (2 / 9)))

    lags = max(0, min(int(lags), n - 1))

    gamma0 = float(np.dot(demeaned, demeaned) / n)
    lrv = gamma0

    for lag in range(1, lags + 1):
        cov = float(np.dot(demeaned[lag:], demeaned[:-lag]) / n)
        weight = 1.0 - lag / (lags + 1.0)
        lrv += 2.0 * weight * cov

    se = math.sqrt(max(lrv, 0) / n) if n > 0 else np.nan
    t = mean / se if se and se > 0 else np.nan

    return mean, t


def block_bootstrap_p_positive(
    x: Sequence[float],
    block: int = BOOTSTRAP_BLOCK,
    n_boot: int = BOOTSTRAP_N,
) -> float:
    arr = pd.Series(x).dropna().astype(float).to_numpy()
    n = len(arr)

    block = max(1, int(block))

    if n < max(10, block):
        return np.nan

    blocks = [arr[i:i + block] for i in range(0, n - block + 1)]
    if not blocks:
        return np.nan

    means = []
    need_blocks = int(math.ceil(n / block))

    for _ in range(n_boot):
        sample = np.concatenate([
            blocks[i]
            for i in rng.integers(0, len(blocks), size=need_blocks)
        ])[:n]
        means.append(np.mean(sample))

    return float(np.mean(np.asarray(means) <= 0))


def summarize_directed_returns(
    directed_y: Sequence[float],
    raw_future_ret: Optional[Sequence[float]] = None,
    k: Optional[int] = None,
    label: str = "",
) -> Dict[str, float]:
    y_raw = pd.to_numeric(
        pd.Series(directed_y).replace([np.inf, -np.inf], np.nan),
        errors="coerce",
    )

    if raw_future_ret is not None:
        raw_raw = pd.to_numeric(
            pd.Series(raw_future_ret).replace([np.inf, -np.inf], np.nan),
            errors="coerce",
        )
        if len(y_raw) != len(raw_raw):
            raise ValueError(f"{label}: directed_y 与 raw_future_ret 长度不一致")
    else:
        raw_raw = None

    y = y_raw.dropna().astype(float)
    n = int(len(y))

    if n == 0:
        return {
            "label": label,
            "K": k,
            "n": 0,
            "mean_y": np.nan,
            "t_nw": np.nan,
            "nw_lags": np.nan,
            "hit_rate": np.nan,
            "boot_block": np.nan,
            "boot_p_mean_le_0": np.nan,
            "mean_raw": np.nan,
            "q05_y": np.nan,
            "q95_y": np.nan,
        }

    if raw_raw is not None:
        raw_aligned = raw_raw.loc[y.index]
        if raw_aligned.isna().any():
            raise ValueError(f"{label}: directed_y 有效样本对应的 raw_future_ret 存在缺失")
        raw = raw_aligned.astype(float)
    else:
        raw = pd.Series(dtype=float)

    k_eff = int(k) if k is not None and pd.notna(k) else 1

    auto_lags = int(np.floor(4 * (n / 100) ** (2 / 9)))
    overlap_lags = max(0, k_eff - 1)
    nw_lags = max(auto_lags, overlap_lags)
    nw_lags = max(0, min(int(nw_lags), n - 1))

    boot_block = max(BOOTSTRAP_BLOCK, k_eff)

    _, t_nw = nw_tstat(y, lags=nw_lags)

    return {
        "label": label,
        "K": k,
        "n": n,
        "mean_y": float(y.mean()),
        "t_nw": float(t_nw) if pd.notna(t_nw) else np.nan,
        "nw_lags": int(nw_lags),
        "hit_rate": float((y > 0).mean()),
        "boot_block": int(boot_block),
        "boot_p_mean_le_0": block_bootstrap_p_positive(y, block=boot_block),
        "mean_raw": float(raw.mean()) if len(raw) else np.nan,
        "q05_y": float(y.quantile(0.05)),
        "q95_y": float(y.quantile(0.95)),
    }


def add_model_flags(scores: pd.DataFrame) -> pd.DataFrame:
    if scores.empty:
        return scores

    out = scores.copy()

    out["candidate_model"] = (
        (out["n"] >= MIN_MODEL_N)
        & (out["mean_y"] > 0)
        & (out["t_nw"] >= MIN_T_CAND)
        & (out["hit_rate"] > MIN_HIT_RATE)
    )

    out["confirmed_model"] = (
        out["candidate_model"]
        & (out["n"] >= MIN_CONFIRMED_N)
        & (out["t_nw"] >= MIN_T_CONF)
        & (out["hit_rate"] > MIN_HIT_RATE)
        & ((out["boot_p_mean_le_0"].isna()) | (out["boot_p_mean_le_0"] <= 0.10))
    )

    out["model_quality"] = np.select(
        [out["confirmed_model"], out["candidate_model"]],
        ["CONFIRMED", "CANDIDATE"],
        default="DIAGNOSTIC_ONLY",
    )

    return out


def zscore_by_commodity(df: pd.DataFrame, col: str) -> pd.Series:
    def _z(s):
        std = s.std(ddof=0)
        return (s - s.mean()) / std if pd.notna(std) and std > 0 else pd.Series(np.nan, index=s.index)

    return df.groupby("commodity")[col].transform(_z)


def pick_best_model(scores: pd.DataFrame) -> pd.DataFrame:
    if scores.empty:
        return pd.DataFrame()

    out = scores.copy()

    for c in ["confirmed_model", "candidate_model"]:
        if c not in out.columns:
            out[c] = False

    out = out.sort_values(
        ["commodity", "confirmed_model", "candidate_model", "t_nw", "mean_y", "hit_rate", "n"],
        ascending=[True, False, False, False, False, False, False],
    )

    return out.groupby("commodity", as_index=False).head(1).reset_index(drop=True)

In [29]:
def add_features_one_commodity(g: pd.DataFrame) -> pd.DataFrame:
    g = g.copy().sort_values("date").reset_index(drop=True)
    g["series_age"] = np.arange(1, len(g) + 1)
    g["series_len"] = len(g)
    g["series_remain"] = g["series_len"] - g["series_age"]

    g["ret_clean"] = np.log(g["close"] / g["close"].shift(1))
    g.loc[g["series_age"] == 1, "ret_clean"] = np.nan

    prev_close = g["close"].shift(1)
    tr1 = g["high"] - g["low"]
    tr2 = (g["high"] - prev_close).abs()
    tr3 = (g["low"] - prev_close).abs()
    g["tr"] = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    g.loc[g["series_age"] == 1, "tr"] = np.nan
    g["atr20"] = g["tr"].rolling(ATR_STOP_LOOKBACK, min_periods=max(5, ATR_STOP_LOOKBACK // 2)).mean()

    g["log_volume"] = np.log1p(g["volume"].clip(lower=0))

    g["sigma_roll20"] = g["ret_clean"].rolling(
        VOL_BASE_LOOKBACK,
        min_periods=max(5, VOL_BASE_LOOKBACK // 2),
    ).std(ddof=0)

    g["sigma_ewma"] = g["ret_clean"].ewm(
        alpha=1 - EWMA_LAMBDA,
        adjust=False,
        min_periods=5,
    ).std()

    g["sigma_daily_base"] = g["sigma_roll20"].combine_first(g["sigma_ewma"])
    g.loc[g["sigma_daily_base"] < MIN_SIGMA, "sigma_daily_base"] = np.nan

    g["pk_var_1"] = (np.log(g["high"] / g["low"]) ** 2) / (4 * np.log(2))
    g["pk_sigma_1"] = np.sqrt(g["pk_var_1"]).replace([np.inf, -np.inf], np.nan)

    g["std_ret_1"] = g["ret_clean"] / g["sigma_daily_base"]
    g["shock_flag"] = (g["std_ret_1"].abs() >= SHOCK_Z_THRESHOLD).astype(float)

    for h in sorted(set(H_RET_LIST + J_LIST)):
        g[f"R_{h}"] = g["ret_clean"].rolling(h, min_periods=h).sum()
        g[f"sigma_base_{h}"] = g["sigma_daily_base"] * math.sqrt(h)
        g[f"D_{h}"] = g[f"R_{h}"] / g[f"sigma_base_{h}"]

        volz_raw = (
            (g["log_volume"] - g["log_volume"].rolling(h, min_periods=h).mean())
            / g["log_volume"].rolling(h, min_periods=h).std(ddof=0)
        )
        g[f"VolZ_{h}"] = volz_raw
        g[f"VolZ_lag_{h}"] = volz_raw.shift(1)

        past_std = g["std_ret_1"].shift(1)
        g[f"RAMOM_{h}"] = past_std.rolling(h, min_periods=h).sum()

        past_ret = g["ret_clean"].shift(1)

        def _mcs(x):
            total = np.nansum(x)
            side = np.sign(total)
            if side == 0 or np.isnan(side):
                return np.nan
            return float(np.mean(np.sign(x) == side))

        g[f"MCS_{h}"] = past_ret.rolling(h, min_periods=h).apply(_mcs, raw=True)

        # 成交量只确认放量；窗口不足时保持 NaN。
        vol_valid = g[f"RAMOM_{h}"].notna() & g[f"VolZ_lag_{h}"].notna()

        g[f"VolumeConfirm_{h}"] = np.nan
        g.loc[vol_valid, f"VolumeConfirm_{h}"] = (
            (g.loc[vol_valid, f"RAMOM_{h}"] != 0)
            & (g.loc[vol_valid, f"VolZ_lag_{h}"] >= VOLUME_CONFIRM_Z)
        ).astype(float)

    for k in K_LIST:
        g[f"fwd_ret_{k}"] = np.log(g["close"].shift(-k) / g["close"])
        g[f"valid_target_{k}"] = (g["series_remain"] >= k).astype(float)
        g.loc[g[f"valid_target_{k}"] != 1, f"fwd_ret_{k}"] = np.nan
        g[f"Y_{k}"] = g[f"fwd_ret_{k}"] / (g["sigma_daily_base"] * math.sqrt(k))

    return g.replace([np.inf, -np.inf], np.nan)


feature_panel = pd.concat(
    [add_features_one_commodity(g) for _, g in panel.groupby("commodity", sort=False)],
    ignore_index=True,
).reset_index(drop=True)

display(feature_panel.head())
display(feature_panel[["commodity", "date", "close", "ret_clean", "sigma_daily_base", "Y_5"]].tail())

,date,datetime,commodity,commodity_name,main_symbol,data_source,open,high,low,close,volume,csv_path,ohlcv_bad,series_age,series_len,series_remain,series_start,series_end,data_reliability,ret_clean,tr,atr20,log_volume,sigma_roll20,sigma_ewma,sigma_daily_base,pk_var_1,pk_sigma_1,std_ret_1,shock_flag,R_1,sigma_base_1,D_1,VolZ_1,VolZ_lag_1,RAMOM_1,MCS_1,VolumeConfirm_1,R_2,sigma_base_2,D_2,VolZ_2,VolZ_lag_2,RAMOM_2,MCS_2,VolumeConfirm_2,R_3,sigma_base_3,D_3,VolZ_3,VolZ_lag_3,RAMOM_3,MCS_3,VolumeConfirm_3,R_5,sigma_base_5,D_5,VolZ_5,VolZ_lag_5,RAMOM_5,MCS_5,VolumeConfirm_5,R_10,sigma_base_10,D_10,VolZ_10,VolZ_lag_10,RAMOM_10,MCS_10,VolumeConfirm_10,R_20,sigma_base_20,D_20,VolZ_20,VolZ_lag_20,RAMOM_20,MCS_20,VolumeConfirm_20,R_40,sigma_base_40,D_40,VolZ_40,VolZ_lag_40,RAMOM_40,MCS_40,VolumeConfirm_40,R_60,sigma_base_60,D_60,VolZ_60,VolZ_lag_60,RAMOM_60,MCS_60,VolumeConfirm_60,fwd_ret_1,valid_target_1,Y_1,fwd_ret_2,valid_target_2,Y_2,fwd_ret_3,valid_target_3,Y_3,fwd_ret_5,valid_target_5,Y_5,fwd_ret_10,valid_target_10,Y_10,fwd_ret_20,valid_target_20,Y_20
0,2016-01-05,2016-01-05,AG,沪银,KQ.m@SHFE.ag,TQ_MAIN_CONTINUOUS_KLINE_DAILY,3306.0,3330.0,3294.0,3313.0,287111.0,main_continuous_daily_trend_momentum_reversal_...,0.0,1,2545,2544,2016-01-05,2026-06-30,MAIN_CONTINUOUS_DIAGNOSTIC,NaN,NaN,NaN,12.567628,NaN,NaN,NaN,0.000043,0.006528,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000302,1.0,NaN,0.013194,1.0,NaN,0.012896,1.0,NaN,-0.007575,1.0,NaN,0.001207,1.0,NaN,0.007518,1.0,NaN
1,2016-01-06,2016-01-06,AG,沪银,KQ.m@SHFE.ag,TQ_MAIN_CONTINUOUS_KLINE_DAILY,3314.0,3321.0,3296.0,3314.0,204526.0,main_continuous_daily_trend_momentum_reversal_...,0.0,2,2545,2543,2016-01-05,2026-06-30,MAIN_CONTINUOUS_DIAGNOSTIC,0.000302,25.0,NaN,12.228455,NaN,NaN,NaN,0.000021,0.004538,NaN,0.0,0.000302,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.012892,1.0,NaN,0.012594,1.0,NaN,0.010506,1.0,NaN,-0.012143,1.0,NaN,0.000302,1.0,NaN,0.008114,1.0,NaN
2,2016-01-07,2016-01-07,AG,沪银,KQ.m@SHFE.ag,TQ_MAIN_CONTINUOUS_KLINE_DAILY,3322.0,3385.0,3308.0,3357.0,488186.0,main_continuous_daily_trend_momentum_reversal_...,0.0,3,2545,2542,2016-01-05,2026-06-30,MAIN_CONTINUOUS_DIAGNOSTIC,0.012892,77.0,NaN,13.098454,NaN,NaN,NaN,0.000191,0.013819,NaN,0.0,0.012892,NaN,NaN,NaN,NaN,NaN,1.0,NaN,0.013194,NaN,NaN,1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,1.304173,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.000298,1.0,NaN,-0.002386,1.0,NaN,-0.020768,1.0,NaN,-0.013495,1.0,NaN,-0.007775,1.0,NaN,0.009192,1.0,NaN
3,2016-01-08,2016-01-08,AG,沪银,KQ.m@SHFE.ag,TQ_MAIN_CONTINUOUS_KLINE_DAILY,3367.0,3390.0,3345.0,3356.0,379737.0,main_continuous_daily_trend_momentum_reversal_...,0.0,4,2545,2541,2016-01-05,2026-06-30,MAIN_CONTINUOUS_DIAGNOSTIC,-0.000298,45.0,NaN,12.847237,NaN,NaN,NaN,0.000064,0.008025,NaN,0.0,-0.000298,NaN,NaN,NaN,NaN,NaN,1.0,NaN,0.012594,NaN,NaN,-1.0,1.0,NaN,1.0,NaN,0.012896,NaN,NaN,0.335135,1.304173,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.002088,1.0,NaN,-0.020470,1.0,NaN,-0.024737,1.0,NaN,-0.018647,1.0,NaN,-0.009882,1.0,NaN,0.009785,1.0,NaN
4,2016-01-11,2016-01-11,AG,沪银,KQ.m@SHFE.ag,TQ_MAIN_CONTINUOUS_KLINE_DAILY,3345.0,3362.0,3322.0,3349.0,283036.0,main_continuous_daily_trend_momentum_reversal_...,0.0,5,2545,2540,2016-01-05,2026-06-30,MAIN_CONTINUOUS_DIAGNOSTIC,-0.002088,40.0,NaN,12.553333,NaN,NaN,NaN,0.000052,0.007

,commodity,date,close,ret_clean,sigma_daily_base,Y_5
24197,V,2026-06-24,4490.0,0.010298,0.010578,NaN
24198,V,2026-06-25,4469.0,-0.004688,0.010426,NaN
24199,V,2026-06-26,4388.0,-0.018291,0.010006,NaN
24200,V,2026-06-29,4391.0,0.000683,0.010092,NaN
24201,V,2026-06-30,4354.0,-0.008462,0.009038,NaN


In [30]:
def feature_target_audit(df: pd.DataFrame) -> Dict[str, Dict]:
    report = {}

    ret_bad = {}
    for h in H_RET_LIST:
        ret_bad[h] = int(df.loc[df["series_age"] < h + 1, f"R_{h}"].notna().sum())
    report["return_feature_non_nan_too_early"] = ret_bad

    momentum_bad = {}
    for h in J_LIST:
        momentum_bad[h] = {
            "RAMOM": int(df.loc[df["series_age"] < h + 2, f"RAMOM_{h}"].notna().sum()),
            "MCS": int(df.loc[df["series_age"] < h + 2, f"MCS_{h}"].notna().sum()),
            "VolZ_lag": int(df.loc[df["series_age"] < h + 1, f"VolZ_lag_{h}"].notna().sum()),
            "VolumeConfirm": int(df.loc[df["series_age"] < h + 2, f"VolumeConfirm_{h}"].notna().sum()),
        }
    report["momentum_feature_non_nan_too_early"] = momentum_bad

    target_bad = {}
    for k in K_LIST:
        target_bad[k] = int(df.loc[df["series_remain"] < k, f"fwd_ret_{k}"].notna().sum())
    report["future_target_non_nan_beyond_series_end"] = target_bad

    bad_ohlc = int((
        (df["open"] <= 0)
        | (df["high"] <= 0)
        | (df["low"] <= 0)
        | (df["close"] <= 0)
        | (df["high"] < df[["open", "close", "low"]].max(axis=1))
        | (df["low"] > df[["open", "close", "high"]].min(axis=1))
    ).sum())
    report["bad_ohlc_after_cleaning"] = {"count": bad_ohlc}

    return report


audit_feature_target = feature_target_audit(feature_panel)
display(audit_feature_target)


def _audit_has_failure(x) -> bool:
    if isinstance(x, dict):
        return any(_audit_has_failure(v) for v in x.values())
    return x != 0


audit_failed = _audit_has_failure(audit_feature_target)

if audit_failed:
    raise RuntimeError(f"Feature / target audit failed: {audit_feature_target}")

{'return_feature_non_nan_too_early': {1: 0,
  2: 0,
  3: 0,
  5: 0,
  10: 0,
  20: 0,
  40: 0,
  60: 0},
 'momentum_feature_non_nan_too_early': {2: {'RAMOM': 0,
   'MCS': 0,
   'VolZ_lag': 0,
   'VolumeConfirm': 0},
  3: {'RAMOM': 0, 'MCS': 0, 'VolZ_lag': 0, 'VolumeConfirm': 0},
  5: {'RAMOM': 0, 'MCS': 0, 'VolZ_lag': 0, 'VolumeConfirm': 0},
  10: {'RAMOM': 0, 'MCS': 0, 'VolZ_lag': 0, 'VolumeConfirm': 0},
  20: {'RAMOM': 0, 'MCS': 0, 'VolZ_lag': 0, 'VolumeConfirm': 0},
  40: {'RAMOM': 0, 'MCS': 0, 'VolZ_lag': 0, 'VolumeConfirm': 0},
  60: {'RAMOM': 0, 'MCS': 0, 'VolZ_lag': 0, 'VolumeConfirm': 0}},
 'future_target_non_nan_beyond_series_end': {1: 0,
  2: 0,
  3: 0,
  5: 0,
  10: 0,
  20: 0},
 'bad_ohlc_after_cleaning': {'count': 0}}

In [31]:
def momentum_score_table(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for commodity, g in df.groupby("commodity", sort=False):
        for j in J_LIST:
            side = np.sign(g[f"RAMOM_{j}"])
            mom_exists = (
                (g[f"RAMOM_{j}"].abs() >= MOM_Z_THRESHOLD)
                & (g[f"MCS_{j}"] >= MCS_THRESHOLD)
            )
            for k in K_LIST:
                mask = (
                    mom_exists
                    & (g[f"valid_target_{k}"] == 1)
                    & g[f"Y_{k}"].notna()
                    & side.notna()
                    & (side != 0)
                )
                if mask.sum() == 0:
                    continue

                stat = summarize_directed_returns(
                    side.loc[mask] * g.loc[mask, f"Y_{k}"],
                    raw_future_ret=side.loc[mask] * g.loc[mask, f"fwd_ret_{k}"],
                    k=k,
                    label=f"{commodity}_Momentum_J{j}_K{k}",
                )
                stat.update({"commodity": commodity, "J": j, "signal_type": "Momentum"})
                rows.append(stat)

    out = pd.DataFrame(rows)
    if out.empty:
        return out

    out = add_model_flags(out)
    return out.sort_values(
        ["commodity", "confirmed_model", "candidate_model", "t_nw"],
        ascending=[True, False, False, False],
    )


momentum_scores = momentum_score_table(feature_panel)
momentum_best = pick_best_model(momentum_scores.rename(columns={"J": "h"})).rename(columns={"h": "J"})

display(momentum_scores.head(12))
display(momentum_best)
momentum_scores.to_csv(RESULT_DIR / "momentum_model_scores.csv", index=False)


,label,K,n,mean_y,t_nw,nw_lags,hit_rate,boot_block,boot_p_mean_le_0,mean_raw,q05_y,q95_y,commodity,J,signal_type,candidate_model,confirmed_model,model_quality
0,AG_Momentum_J2_K1,1,800,0.106057,2.437516,6,0.518750,5,0.0025,0.000735,-1.767135,2.444140,AG,2,Momentum,True,True,CONFIRMED
1,AG_Momentum_J2_K2,2,800,0.091262,1.978080,6,0.503750,5,0.0350,0.000884,-1.735145,2.211434,AG,2,Momentum,True,False,CANDIDATE
37,AG_Momentum_J60_K2,2,405,0.136471,1.433764,5,0.555556,5,0.0800,0.003910,-1.810688,2.095758,AG,60,Momentum,True,False,CANDIDATE
36,AG_Momentum_J60_K1,1,405,0.084798,1.196110,5,0.535802,5,0.1100,0.001806,-2.073653,2.398445,AG,60,Momentum,True,False,CANDIDATE
29,AG_Momentum_J20_K20,20,906,0.169699,1.155607,19,0.508830,20,0.1000,0.013517,-1.585439,2.692946,AG,20,Momentum,True,False,CANDIDATE
32,AG_Momentum_J40_K3,3,411,0.117805,1.154943,5,0.503650,5,0.1075,0.004529,-1.799903,2.325392,AG,40,Momentum,True,False,CANDIDATE
33,AG_Momentum_J40_K5,5,411,0.141954,1.136861,5,0.508516,5,0.1225,0.007506,-1.864183,2.421192,AG,40,Momentum,True,False,CANDIDATE
30,AG_Momentum_J40_K1,1,411,0.072049,1.103102,5,0.527981,5,0.1575,0.001487,-1.841262,2.230260,AG,40,Momentum,True,False,CANDIDATE
38,AG_Momentum_J60_K3,3,405,0.126716,1.102740,5,0.545679,5,0.1400,0.004879,-2.037340,2.336621,AG,60,Momentum,True,False,CANDIDATE
31,AG_Momentum_J40_K2,2,411,0.092599,1.077860,5,0.537713,5,0.1150,0.002956,-1.775275,2.100583,AG,40,Momentum,True,False,CANDIDATE


,label,K,n,mean_y,t_nw,nw_lags,hit_rate,boot_block,boot_p_mean_le_0,mean_raw,q05_y,q95_y,commodity,J,signal_type,candidate_model,confirmed_model,model_quality
0,AG_Momentum_J2_K1,1,800,0.106057,2.437516,6,0.518750,5,0.0025,0.000735,-1.767135,2.444140,AG,2,Momentum,True,True,CONFIRMED
1,CF_Momentum_J3_K2,2,1209,0.138725,3.200463,6,0.510339,5,0.0000,0.001790,-1.484554,2.144700,CF,3,Momentum,True,True,CONFIRMED
2,EB_Momentum_J20_K5,5,722,0.225588,2.361195,6,0.540166,5,0.0025,0.007204,-1.684649,2.345782,EB,20,Momentum,True,True,CONFIRMED
3,FG_Momentum_J40_K1,1,510,0.114226,2.129536,5,0.537255,5,0.0175,0.001370,-1.808319,2.089731,FG,40,Momentum,True,True,CONFIRMED
4,LH_Momentum_J20_K5,5,569,0.232740,1.558231,5,0.548330,5,0.0575,0.005783,-1.988317,3.059507,LH,20,Momentum,True,False,CANDIDATE
5,M_Momentum_J3_K20,20,1225,0.062819,1.360382,19,0.515918,20,0.0675,0.003155,-1.963766,2.179560,M,3,Momentum,True,False,CANDIDATE
6,MA_Momentum_J3_K20,20,1292,0.033127,0.732274,19,0.503870,20,0.3250,0.002264,-1.678233,1.960788,MA,3,Momentum,False,False,DIAGNOSTIC_ONLY
7,SA_Momentum_J2_K2,2,524,0.155320,2.357919,5,0.549618,5,0.0150,0.003686,-1.653538,2.086934,SA,2,Momentum,True,True,CONFIRMED
8,SP_Momentum_J40_K20,20,357,0.577768,3.407249,19,0.666667,20,0.0000,0.031221,-1.219468,2.603318,SP,40,Momentum,True,True,CONFIRMED
9,SR_Momentum_J2_K2,2,807,0.061641,1.337171,6,0.488228,5,0.0800,0.000458,-1.752761,2.029387,SR,2,Momentum,False,False,DIAGNOSTIC_ONLY


In [32]:
MOM_ESTABLISH_CONFIRM_DAYS = globals().get("MOM_ESTABLISH_CONFIRM_DAYS", 2)
MOM_EXTREME_Q = globals().get("MOM_EXTREME_Q", 0.90)
MOM_EXTREME_MIN_PERIODS = globals().get("MOM_EXTREME_MIN_PERIODS", 120)


def is_usable_momentum_model(row) -> bool:
    def _as_bool(x) -> bool:
        return bool(x) if pd.notna(x) else False

    quality = str(row.get("model_quality", "DIAGNOSTIC_ONLY")).upper()
    return (
        _as_bool(row.get("confirmed_model", False))
        or _as_bool(row.get("candidate_model", False))
        or quality in {"CONFIRMED", "CANDIDATE"}
    )


def add_momentum_state_columns(df: pd.DataFrame, best: pd.DataFrame) -> pd.DataFrame:
    out = df.copy().sort_values(["commodity", "date"]).reset_index(drop=True)

    state_cols = [
        "J_mom_star", "K_mom_eval", "mom_confirmed_model", "mom_candidate_model",
        "mom_model_scope", "MOMModelUsable", "MOMMain", "MOMMainSide",
        "MOMMainExists", "MOMEstablished", "MOMValidatedEstablished", "MOMStart",
        "MOMMainMCS", "MOMPathConfirm", "MOMVolumeConfirm", "MOMExtremeThr",
        "MOMExtreme", "MOMRiskUp", "MOMEndRisk", "MOMEnd_loss", "MOMEnd_flip",
        "MOMEnd_signal", "MOMEnd_path", "MOMEnd_volume", "MOMEndConfirmed",
        "MOMEnd", "MOMRawStartDate",
    ]
    for col in state_cols:
        out[col] = np.nan

    out["mom_confirmed_model"] = False
    out["mom_candidate_model"] = False
    out["MOMModelUsable"] = False
    out["mom_model_scope"] = pd.Series("NO_MODEL", index=out.index, dtype="object")

    if best is not None and not best.empty:
        for _, row in best.iterrows():
            commodity = row["commodity"]
            j = int(row["J"])
            k = int(row["K"])
            m = out["commodity"] == commodity

            out.loc[m, "J_mom_star"] = j
            out.loc[m, "K_mom_eval"] = k
            out.loc[m, "mom_confirmed_model"] = bool(row.get("confirmed_model", False))
            out.loc[m, "mom_candidate_model"] = bool(row.get("candidate_model", False))
            out.loc[m, "mom_model_scope"] = row.get("model_quality", "DIAGNOSTIC_ONLY")
            out.loc[m, "MOMModelUsable"] = is_usable_momentum_model(row)

            out.loc[m, "MOMMain"] = out.loc[m, f"RAMOM_{j}"]
            out.loc[m, "MOMMainSide"] = np.sign(out.loc[m, "MOMMain"])
            out.loc[m, "MOMMainMCS"] = out.loc[m, f"MCS_{j}"]
            out.loc[m, "MOMPathConfirm"] = (out.loc[m, "MOMMainMCS"] >= MCS_THRESHOLD).astype(float)
            out.loc[m, "MOMVolumeConfirm"] = out.loc[m, f"VolumeConfirm_{j}"]

            out.loc[m, "MOMMainExists"] = (
                (out.loc[m, "MOMMain"].abs() >= MOM_Z_THRESHOLD)
                & (out.loc[m, "MOMPathConfirm"] == 1)
                & out.loc[m, "MOMMainSide"].notna()
                & (out.loc[m, "MOMMainSide"] != 0)
            ).astype(float)

    frames = []
    fixed_extreme_thr = MOM_Z_THRESHOLD + 1.0

    for _, g in out.groupby("commodity", sort=False):
        g = g.copy().sort_values("date").reset_index(drop=True)

        active = (g["MOMMainExists"] == 1) & g["MOMMainSide"].notna() & (g["MOMMainSide"] != 0)
        side = g["MOMMainSide"]
        block = ((~active) | (side != side.shift(1))).cumsum()
        run_len = active.astype(int).groupby(block).cumsum()

        g["MOMEstablished"] = (active & (run_len >= MOM_ESTABLISH_CONFIRM_DAYS)).astype(float)
        g["MOMValidatedEstablished"] = (
            (g["MOMEstablished"] == 1)
            & (g["MOMModelUsable"] == True)
        ).astype(float)

        prev_established = g["MOMEstablished"].shift(1).fillna(0).eq(1)
        prev_side = g["MOMMainSide"].shift(1)
        g["MOMStart"] = ((g["MOMEstablished"] == 1) & ~prev_established).astype(float)

        raw_start_dates = pd.Series(pd.NaT, index=g.index, dtype="datetime64[ns]")
        for _, idx in g.index[active].to_series().groupby(block[active]).groups.items():
            idx = list(idx)
            raw_start_dates.loc[idx] = g.loc[idx[0], "date"]
        g["MOMRawStartDate"] = raw_start_dates

        hist_q = (
            g["MOMMain"].abs()
            .expanding(min_periods=MOM_EXTREME_MIN_PERIODS)
            .quantile(MOM_EXTREME_Q)
            .shift(1)
        )
        g["MOMExtremeThr"] = hist_q.clip(lower=fixed_extreme_thr).fillna(fixed_extreme_thr)

        g["MOMExtreme"] = (
            (g["MOMEstablished"] == 1)
            & (g["MOMMain"].abs() >= g["MOMExtremeThr"])
        ).astype(float)
        g["MOMRiskUp"] = g["MOMExtreme"]
        g["MOMEndRisk"] = g["MOMExtreme"]

        g["MOMEnd_loss"] = (prev_established & (g["MOMMain"].abs() < MOM_Z_THRESHOLD)).astype(float)
        g["MOMEnd_flip"] = (
            prev_established
            & g["MOMMainSide"].notna()
            & prev_side.notna()
            & (np.sign(g["MOMMainSide"]) == -np.sign(prev_side))
        ).astype(float)
        g["MOMEnd_signal"] = (prev_established & (g["MOMMainExists"] != 1)).astype(float)
        g["MOMEnd_path"] = (prev_established & (g["MOMPathConfirm"] != 1)).astype(float)
        g["MOMEnd_volume"] = (prev_established & (g["MOMVolumeConfirm"] == 0)).astype(float)

        g["MOMEndConfirmed"] = (
            (g["MOMEnd_loss"] == 1)
            | (g["MOMEnd_flip"] == 1)
            | (g["MOMEnd_signal"] == 1)
            | (g["MOMEnd_path"] == 1)
        ).astype(float)
        g["MOMEnd"] = g["MOMEndConfirmed"]

        frames.append(g)

    return pd.concat(frames, ignore_index=True).replace([np.inf, -np.inf], np.nan)


panel_state = add_momentum_state_columns(feature_panel, momentum_best)

display(panel_state[[
    "date", "commodity", "J_mom_star", "K_mom_eval", "mom_model_scope",
    "MOMModelUsable", "MOMMain", "MOMMainSide", "MOMMainExists",
    "MOMEstablished", "MOMValidatedEstablished", "MOMExtreme",
    "MOMEndRisk", "MOMEndConfirmed",
]].tail(24))

,date,commodity,J_mom_star,K_mom_eval,mom_model_scope,MOMModelUsable,MOMMain,MOMMainSide,MOMMainExists,MOMEstablished,MOMValidatedEstablished,MOMExtreme,MOMEndRisk,MOMEndConfirmed
24178,2026-05-27,V,3.0,1.0,CONFIRMED,True,-0.134824,-1.0,0.0,0.0,0.0,0.0,0.0,0.0
24179,2026-05-28,V,3.0,1.0,CONFIRMED,True,0.355547,1.0,0.0,0.0,0.0,0.0,0.0,0.0
24180,2026-05-29,V,3.0,1.0,CONFIRMED,True,0.319765,1.0,0.0,0.0,0.0,0.0,0.0,0.0
24181,2026-06-01,V,3.0,1.0,CONFIRMED,True,0.886526,1.0,0.0,0.0,0.0,0.0,0.0,0.0
24182,2026-06-02,V,3.0,1.0,CONFIRMED,True,1.572382,1.0,1.0,0.0,0.0,0.0,0.0,0.0
24183,2026-06-03,V,3.0,1.0,CONFIRMED,True,0.145228,1.0,0.0,0.0,0.0,0.0,0.0,0.0
24184,2026-06-04,V,3.0,1.0,CONFIRMED,True,0.258538,1.0,0.0,0.0,0.0,0.0,0.0,0.0
24185,2026-06-05,V,3.0,1.0,CONFIRMED,True,-2.440535,-1.0,1.0,0.0,0.0,0.0,0.0,0.0
24186,2026-06-08,V,3.0,1.0,CONFIRMED,True,-2.613233,-1.0,1.0,1.0,1.0,0.0,0.0,0.0
24187,2026-06-09,V,3.0,1.0,CONFIRMED,True,-2.358521,-1.0,1.0,1.0,1.0,0.0,0.0,0.0


In [33]:
MOM_SURVIVAL_HORIZONS = sorted(set(globals().get("MOM_SURVIVAL_HORIZONS", K_LIST)))
MOM_DURATION_MIN_COMPLETED = globals().get("MOM_DURATION_MIN_COMPLETED", 5)


def _mom_side_label(side) -> str:
    if pd.isna(side) or side == 0:
        return "NONE"
    return "UP" if side > 0 else "DOWN"


def _mom_end_reason(row: pd.Series) -> str:
    if bool(row.get("MOMEnd_flip", False)):
        return "FLIP"
    if bool(row.get("MOMEnd_loss", False)):
        return "LOSS"
    if bool(row.get("MOMEnd_path", False)):
        return "PATH_LOSS"
    if bool(row.get("MOMEnd_signal", False)):
        return "SIGNAL_LOSS"
    return "UNFLAGGED"


def build_momentum_episodes(state: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    required = [
        "date", "commodity", "series_remain", "MOMMain", "MOMMainSide",
        "MOMMainExists", "MOMEstablished", "MOMValidatedEstablished",
        "MOMEndRisk", "MOMEndConfirmed", "J_mom_star", "K_mom_eval",
        "mom_model_scope",
    ]
    missing = [c for c in required if c not in state.columns]
    if missing:
        raise RuntimeError(f"Momentum state is missing columns: {missing}")

    out = state.copy().sort_values(["commodity", "date"]).reset_index(drop=True)
    for col in [
        "MOMEpisodeID", "MOMEpisodeSide", "MOMEpisodeSideLabel", "MOMAge",
        "MOMEpisodeStartDate", "MOMEpisodeRawStartDate", "MOMEpisodeEndDate",
        "MOMEpisodeEndReason", "MOMEpisodeCensored", "MOMDaysToEnd",
    ]:
        out[col] = pd.NA

    episodes = []

    def close_episode(g, meta, end_row, end_reason, censored):
        idxs = meta["idxs"]
        if not idxs:
            return

        n = len(idxs)
        end_date = pd.NaT if censored else end_row["date"]

        for age, local_idx in enumerate(idxs, start=1):
            global_idx = g.loc[local_idx, "_global_idx"]
            out.loc[global_idx, "MOMEpisodeID"] = meta["episode_id"]
            out.loc[global_idx, "MOMEpisodeSide"] = meta["side"]
            out.loc[global_idx, "MOMEpisodeSideLabel"] = _mom_side_label(meta["side"])
            out.loc[global_idx, "MOMAge"] = age
            out.loc[global_idx, "MOMEpisodeStartDate"] = meta["start_date"]
            out.loc[global_idx, "MOMEpisodeRawStartDate"] = meta["raw_start_date"]
            out.loc[global_idx, "MOMEpisodeEndDate"] = end_date
            out.loc[global_idx, "MOMEpisodeEndReason"] = end_reason
            out.loc[global_idx, "MOMEpisodeCensored"] = bool(censored)
            if not censored:
                out.loc[global_idx, "MOMDaysToEnd"] = n - age + 1

        seg = g.loc[idxs]

        episodes.append({
            "commodity": meta["commodity"],
            "mom_episode_id": meta["episode_id"],
            "mom_side": meta["side"],
            "mom_side_label": _mom_side_label(meta["side"]),
            "raw_start_date": meta["raw_start_date"],
            "established_date": meta["start_date"],
            "last_established_date": seg["date"].max(),
            "end_signal_date": end_date,
            "end_reason": end_reason,
            "active_duration_days": n,
            "duration_to_end_signal_days": n if not censored else np.nan,
            "is_censored": bool(censored),
            "ever_end_risk": bool((seg["MOMEndRisk"] == 1).any()),
            "first_end_risk_date": (
                seg.loc[seg["MOMEndRisk"] == 1, "date"].min()
                if (seg["MOMEndRisk"] == 1).any()
                else pd.NaT
            ),
            "max_abs_mom": float(seg["MOMMain"].abs().max()),
            "mean_abs_mom": float(seg["MOMMain"].abs().mean()),
            "mean_mcs": float(seg["MOMMainMCS"].mean()) if "MOMMainMCS" in seg else np.nan,
            "J_mom_star": seg["J_mom_star"].dropna().iloc[-1] if seg["J_mom_star"].notna().any() else np.nan,
            "K_mom_eval": seg["K_mom_eval"].dropna().iloc[-1] if seg["K_mom_eval"].notna().any() else np.nan,
            "mom_model_scope": (
                seg["mom_model_scope"].dropna().iloc[-1]
                if seg["mom_model_scope"].notna().any()
                else "NO_MODEL"
            ),
        })

    for commodity, g0 in out.groupby("commodity", sort=False):
        g = g0.copy().reset_index().rename(columns={"index": "_global_idx"})
        open_ep = None
        ep_no = 0

        for i, row in g.iterrows():
            established = bool(row.get("MOMValidatedEstablished", 0) == 1)
            side = row.get("MOMMainSide", np.nan)
            valid_side = pd.notna(side) and side != 0

            if open_ep is not None:
                should_close = (
                    bool(row.get("MOMEndConfirmed", 0) == 1)
                    or (not established)
                    or (not valid_side)
                    or (np.sign(side) != np.sign(open_ep["side"]))
                )
                if should_close:
                    close_episode(g, open_ep, row, _mom_end_reason(row), censored=False)
                    open_ep = None

            if open_ep is None and established and valid_side:
                ep_no += 1
                raw_start = row.get("MOMRawStartDate", pd.NaT)
                if pd.isna(raw_start):
                    raw_start = row["date"]

                open_ep = {
                    "commodity": commodity,
                    "episode_id": f"{commodity}_MOM_{ep_no:04d}",
                    "side": float(np.sign(side)),
                    "start_date": row["date"],
                    "raw_start_date": raw_start,
                    "idxs": [],
                }

            if open_ep is not None and established and valid_side and (np.sign(side) == np.sign(open_ep["side"])):
                open_ep["idxs"].append(i)

        if open_ep is not None:
            close_episode(g, open_ep, g.iloc[-1], "END_OF_SAMPLE", censored=True)

    return out.replace([np.inf, -np.inf], np.nan), pd.DataFrame(episodes).replace([np.inf, -np.inf], np.nan)


def summarize_momentum_durations(episodes: pd.DataFrame) -> pd.DataFrame:
    if episodes.empty:
        return pd.DataFrame()

    rows = []

    for keys, g in episodes.groupby(["commodity", "mom_side_label"], sort=False):
        commodity, side_label = keys
        ended = g.loc[~g["is_censored"].astype(bool)].copy()

        rows.append({
            "commodity": commodity,
            "mom_side_label": side_label,
            "n_episodes": int(len(g)),
            "n_ended": int(len(ended)),
            "n_censored": int(g["is_censored"].sum()),
            "end_risk_rate": float(g["ever_end_risk"].mean()) if len(g) else np.nan,
            "mean_active_duration": float(ended["active_duration_days"].mean()) if len(ended) else np.nan,
            "median_active_duration": float(ended["active_duration_days"].median()) if len(ended) else np.nan,
            "q25_active_duration": float(ended["active_duration_days"].quantile(0.25)) if len(ended) else np.nan,
            "q75_active_duration": float(ended["active_duration_days"].quantile(0.75)) if len(ended) else np.nan,
            "mean_duration_to_end_signal": float(ended["duration_to_end_signal_days"].mean()) if len(ended) else np.nan,
            "median_duration_to_end_signal": float(ended["duration_to_end_signal_days"].median()) if len(ended) else np.nan,
        })

    return pd.DataFrame(rows)


def momentum_survival_by_horizon(panel_duration: pd.DataFrame) -> pd.DataFrame:
    active = panel_duration.loc[panel_duration["MOMEpisodeID"].notna()].copy()
    if active.empty:
        return pd.DataFrame()

    active["risk_state"] = np.where(active["MOMEndRisk"] == 1, "RISK", "NO_RISK")

    rows = []

    for keys, g in active.groupby(["commodity", "MOMEpisodeSideLabel", "risk_state"], sort=False):
        commodity, side_label, risk_state = keys

        for h in MOM_SURVIVAL_HORIZONS:
            known = g["MOMDaysToEnd"].notna() | (g["series_remain"] >= h)
            gg = g.loc[known]

            if gg.empty:
                continue

            days_to_end = pd.to_numeric(gg["MOMDaysToEnd"], errors="coerce")
            ended = days_to_end.notna() & (days_to_end <= h)

            rows.append({
                "commodity": commodity,
                "mom_side_label": side_label,
                "risk_state": risk_state,
                "horizon": h,
                "n_state_obs": int(len(gg)),
                "n_end_within_h": int(ended.sum()),
                "p_end_within_h": float(ended.mean()),
                "p_survive_beyond_h": float(1 - ended.mean()),
            })

    return pd.DataFrame(rows)


def current_momentum_life_report(panel_duration: pd.DataFrame, episodes: pd.DataFrame) -> pd.DataFrame:
    latest = panel_duration.sort_values(["commodity", "date"]).groupby("commodity", sort=False).tail(1)
    completed = (
        episodes.loc[~episodes.get("is_censored", pd.Series(dtype=bool)).astype(bool)].copy()
        if not episodes.empty
        else pd.DataFrame()
    )

    rows = []

    for _, row in latest.iterrows():
        commodity = row["commodity"]
        established = bool(row.get("MOMValidatedEstablished", 0) == 1)
        side_label = (
            row.get("MOMEpisodeSideLabel", "NONE")
            if established
            else _mom_side_label(row.get("MOMMainSide", np.nan))
        )
        age = row.get("MOMAge", np.nan)

        out = {
            "commodity": commodity,
            "date": row["date"],
            "momentum_established_now": established,
            "mom_side_label": side_label,
            "mom_age": age,
            "mom_end_risk": bool(row.get("MOMEndRisk", 0) == 1),
            "mom_end_confirmed": bool(row.get("MOMEndConfirmed", 0) == 1),
            "n_comparable_completed": 0,
            "duration_reliability": "NO_ACTIVE_MOMENTUM" if not established else "INSUFFICIENT",
            "mean_remaining_days": np.nan,
            "median_remaining_days": np.nan,
        }

        for h in MOM_SURVIVAL_HORIZONS:
            out[f"p_end_within_{h}d"] = np.nan

        if established and pd.notna(age) and not completed.empty:
            comp = completed.loc[
                (completed["commodity"] == commodity)
                & (completed["mom_side_label"] == side_label)
                & (completed["active_duration_days"] >= age)
            ].copy()

            remaining = comp["duration_to_end_signal_days"] - (float(age) - 1)
            remaining = remaining.loc[remaining >= 1].astype(float)

            out["n_comparable_completed"] = int(len(remaining))

            if len(remaining) >= MOM_DURATION_MIN_COMPLETED:
                out["duration_reliability"] = "OK"
                out["mean_remaining_days"] = float(remaining.mean())
                out["median_remaining_days"] = float(remaining.median())

                for h in MOM_SURVIVAL_HORIZONS:
                    out[f"p_end_within_{h}d"] = float((remaining <= h).mean())

        rows.append(out)

    return pd.DataFrame(rows)


panel_state_duration, momentum_episode_table = build_momentum_episodes(panel_state)
momentum_duration_summary = summarize_momentum_durations(momentum_episode_table)
momentum_survival_table = momentum_survival_by_horizon(panel_state_duration)
current_momentum_life_report_df = current_momentum_life_report(panel_state_duration, momentum_episode_table)

display(momentum_episode_table.tail(12))
display(momentum_duration_summary)
display(momentum_survival_table.head(20))
display(current_momentum_life_report_df)

panel_state_duration.to_csv(RESULT_DIR / "momentum_panel_state_duration.csv", index=False)
momentum_episode_table.to_csv(RESULT_DIR / "momentum_episode_table.csv", index=False)
momentum_duration_summary.to_csv(RESULT_DIR / "momentum_duration_summary.csv", index=False)
momentum_survival_table.to_csv(RESULT_DIR / "momentum_survival_table.csv", index=False)
current_momentum_life_report_df.to_csv(RESULT_DIR / "momentum_current_life_report.csv", index=False)

,commodity,mom_episode_id,mom_side,mom_side_label,raw_start_date,established_date,last_established_date,end_signal_date,end_reason,active_duration_days,duration_to_end_signal_days,is_censored,ever_end_risk,first_end_risk_date,max_abs_mom,mean_abs_mom,mean_mcs,J_mom_star,K_mom_eval,mom_model_scope
1418,V,V_MOM_0300,-1.0,DOWN,2026-02-09,2026-02-10,2026-02-11,2026-02-12,FLIP,2,2.0,False,False,NaT,2.423510,1.812411,0.666667,3.0,1.0,CONFIRMED
1419,V,V_MOM_0301,-1.0,DOWN,2026-03-02,2026-03-03,2026-03-03,2026-03-04,FLIP,1,1.0,False,False,NaT,1.391030,1.391030,0.666667,3.0,1.0,CONFIRMED
1420,V,V_MOM_0302,1.0,UP,2026-03-04,2026-03-05,2026-03-12,2026-03-13,LOSS,6,6.0,False,True,2026-03-05,5.230661,3.282798,0.888889,3.0,1.0,CONFIRMED
1421,V,V_MOM_0303,1.0,UP,2026-03-16,2026-03-17,2026-03-18,2026-03-19,LOSS,2,2.0,False,False,NaT,2.081212,2.072976,1.000000,3.0,1.0,CONFIRMED
1422,V,V_MOM_0304,-1.0,DOWN,2026-03-27,2026-03-30,2026-03-30,2026-03-31,LOSS,1,1.0,False,False,NaT,1.283816,1.283816,1.000000,3.0,1.0,CONFIRMED
1423,V,V_MOM_0305,-1.0,DOWN,2026-04-01,2026-04-02,2026-04-02,2026-04-03,LOSS,1,1.0,False,False,NaT,1.988358,1.988358,1.000000,3.0,1.0,CONFIRMED
1424,V,V_MOM_0306,-1.0,DOWN,2026-04-09,2026-04-10,2026-04-14,2026-04-15,LOSS,3,3.0,False,False,NaT,2.790383,2.029607,0.888889,3.0,1.0,CONFIRMED
1425,V,V_MOM_0307,1.0,UP,2026-04-29,2026-04-30,2026-04-30,2026-05-06,LOSS,1,1.0,False,False,NaT,1.307543,1.307543,0.666667,3.0,1.0,CONFIRMED
1426,V,V_MOM_0308,-1.0,DOWN,2026-05-08,2026-05-11,2026-05-12,2026-05-13,LOSS,2,2.0,False,False,NaT,2.125832,1.763995,0.833333,3.0,1.0,CONFIRMED
1427,V,V_MOM_0309,-1.0,DOWN,2026-05-22,2026-05-25,2026-05-25,2026-05-26,LOSS,1,1.0,False,False,NaT,2.582065,2.582065,1.000000,3.0,1.0,CONFIRMED


,commodity,mom_side_label,n_episodes,n_ended,n_censored,end_risk_rate,mean_active_duration,median_active_duration,q25_active_duration,q75_active_duration,mean_duration_to_end_signal,median_duration_to_end_signal
0,AG,UP,102,102,0,0.480392,1.686275,1.0,1.00,2.00,1.686275,1.0
1,AG,DOWN,88,88,0,0.477273,1.488636,1.0,1.00,2.00,1.488636,1.0
2,CF,UP,156,156,0,0.410256,2.314103,2.0,1.00,3.00,2.314103,2.0
3,CF,DOWN,144,144,0,0.395833,2.194444,2.0,1.00,3.00,2.194444,2.0
4,EB,DOWN,33,32,1,0.424242,9.125000,7.5,3.00,13.00,9.125000,7.5
5,EB,UP,36,36,0,0.305556,8.750000,7.0,1.00,10.50,8.750000,7.0
6,FG,UP,20,20,0,0.500000,15.400000,8.0,1.00,20.25,15.400000,8.0
7,FG,DOWN,19,19,0,0.473684,7.684211,6.0,2.50,9.50,7.684211,6.0
8,LH,UP,24,24,0,0.250000,5.625000,4.0,1.75,7.75,5.625000,4.0
9,LH,DOWN,33,33,0,0.303030,10.878788,5.0,2.00,16.00,10.878788,5.0


,commodity,mom_side_label,risk_state,horizon,n_state_obs,n_end_within_h,p_end_within_h,p_survive_beyond_h
0,AG,UP,RISK,1,67,33,0.492537,0.507463
1,AG,UP,RISK,2,67,54,0.805970,0.194030
2,AG,UP,RISK,3,67,62,0.925373,0.074627
3,AG,UP,RISK,5,67,67,1.000000,0.000000
4,AG,UP,RISK,10,67,67,1.000000,0.000000
5,AG,UP,RISK,20,67,67,1.000000,0.000000
6,AG,DOWN,RISK,1,55,34,0.618182,0.381818
7,AG,DOWN,RISK,2,55,50,0.909091,0.090909
8,AG,DOWN,RISK,3,55,54,0.981818,0.018182
9,AG,DOWN,RISK,5,55,55,1.000000,0.000000


,commodity,date,momentum_established_now,mom_side_label,mom_age,mom_end_risk,mom_end_confirmed,n_comparable_completed,duration_reliability,mean_remaining_days,median_remaining_days,p_end_within_1d,p_end_within_2d,p_end_within_3d,p_end_within_5d,p_end_within_10d,p_end_within_20d
0,AG,2026-06-30,False,UP,<NA>,False,False,0,NO_ACTIVE_MOMENTUM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,CF,2026-06-30,False,UP,<NA>,False,False,0,NO_ACTIVE_MOMENTUM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,EB,2026-06-30,True,DOWN,10,True,False,12,OK,8.750000,5.5,0.083333,0.083333,0.250000,0.500000,0.75,0.833333
3,FG,2026-06-30,False,DOWN,<NA>,False,True,0,NO_ACTIVE_MOMENTUM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,LH,2026-06-30,False,UP,<NA>,False,False,0,NO_ACTIVE_MOMENTUM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,M,2026-06-30,True,UP,2,False,False,108,OK,2.037037,1.0,0.564815,0.731481,0.842593,0.944444,1.00,1.000000
6,MA,2026-06-30,False,UP,<NA>,False,False,0,NO_ACTIVE_MOMENTUM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,SA,2026-06-30,False,DOWN,<NA>,False,True,0,NO_ACTIVE_MOMENTUM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,SP,2026-06-30,False,DOWN,<NA>,False,False,0,NO_ACTIVE_MOMENTUM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,SR,2026-06-30,False,UP,<NA>,False,False,0,NO_ACTIVE_MOMENTUM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [36]:
REQUIRED_MOM_FINAL_OBJECTS = [
    "panel_state_duration",
    "momentum_best",
    "momentum_duration_summary",
    "momentum_survival_table",
    "current_momentum_life_report_df",
]
missing = [x for x in REQUIRED_MOM_FINAL_OBJECTS if x not in globals()]
if missing:
    raise RuntimeError(f"Missing required objects. Run previous cells first: {missing}")


def _select(df: pd.DataFrame, cols: list) -> pd.DataFrame:
    return df[[c for c in cols if c in df.columns]].copy()


def _p_end_key(col: str) -> int:
    return int(col.replace("p_end_within_", "").replace("d", ""))


def _reliability(n) -> str:
    if pd.isna(n) or n < 5:
        return "INSUFFICIENT"
    if n < 20:
        return "LOW"
    if n < 50:
        return "MEDIUM"
    return "HIGH"


def _mom_status(row: pd.Series) -> str:
    if bool(row.get("mom_end_confirmed", False)):
        return "MOMENTUM_CONFIRMED_ENDED"
    if not bool(row.get("momentum_established_now", False)):
        return "NO_ESTABLISHED_MOMENTUM"
    side = row.get("mom_side_label", "NONE")
    return f"{side}_MOMENTUM_WITH_END_RISK" if bool(row.get("mom_end_risk", False)) else f"{side}_MOMENTUM_ESTABLISHED"


latest = (
    panel_state_duration
    .sort_values(["commodity", "date"])
    .groupby("commodity", sort=False)
    .tail(1)
    .copy()
)

p_cols = sorted(
    [c for c in current_momentum_life_report_df.columns if c.startswith("p_end_within_")],
    key=_p_end_key,
)

life = _select(current_momentum_life_report_df, [
    "commodity", "momentum_established_now", "mom_side_label", "mom_age",
    "mom_end_risk", "mom_end_confirmed", "n_comparable_completed",
    "duration_reliability", "mean_remaining_days", "median_remaining_days",
] + p_cols)

best = _select(momentum_best, [
    "commodity", "label", "J", "K", "n", "mean_y", "t_nw", "hit_rate",
    "boot_p_mean_le_0", "mean_raw", "candidate_model", "confirmed_model", "model_quality",
]).rename(columns={
    "label": "best_model_label",
    "J": "best_J",
    "K": "best_K",
    "n": "best_n",
    "mean_y": "best_mean_y",
    "t_nw": "best_t_nw",
    "hit_rate": "best_hit_rate",
    "boot_p_mean_le_0": "best_boot_p_mean_le_0",
    "mean_raw": "best_mean_raw",
    "candidate_model": "best_candidate_model",
    "confirmed_model": "best_confirmed_model",
    "model_quality": "best_model_quality",
})

duration = momentum_duration_summary.copy().rename(columns={
    "n_episodes": "hist_n_episodes",
    "n_ended": "hist_n_ended",
    "n_censored": "hist_n_censored",
    "end_risk_rate": "hist_end_risk_rate",
    "mean_active_duration": "hist_mean_active_duration",
    "median_active_duration": "hist_median_active_duration",
    "q25_active_duration": "hist_q25_active_duration",
    "q75_active_duration": "hist_q75_active_duration",
    "mean_duration_to_end_signal": "hist_mean_duration_to_end_signal",
    "median_duration_to_end_signal": "hist_median_duration_to_end_signal",
})
if not duration.empty and "hist_n_ended" in duration.columns:
    duration["hist_duration_reliability"] = duration["hist_n_ended"].apply(_reliability)

q1_current = _select(latest, [
    "commodity", "commodity_name", "main_symbol", "date", "close",
    "J_mom_star", "K_mom_eval", "mom_model_scope",
    "MOMModelUsable", "MOMMain", "MOMMainSide", "MOMMainMCS",
    "MOMPathConfirm", "MOMVolumeConfirm", "MOMMainExists",
    "MOMEstablished", "MOMValidatedEstablished",
    "MOMEpisodeID", "MOMEpisodeSideLabel", "MOMAge",
    "MOMEpisodeStartDate", "MOMEpisodeRawStartDate",
]).merge(best, on="commodity", how="left")

q2_current = _select(latest, [
    "commodity", "commodity_name", "main_symbol", "date", "close",
    "J_mom_star", "K_mom_eval", "mom_model_scope", "MOMModelUsable",
    "MOMEpisodeID", "MOMEpisodeSideLabel", "MOMAge",
    "MOMEpisodeStartDate", "MOMEpisodeRawStartDate", "MOMEpisodeCensored",
]).merge(life, on="commodity", how="left")

q2_current["mom_side_for_duration"] = q2_current["mom_side_label"].where(
    q2_current["momentum_established_now"].fillna(False)
)
q2_current = q2_current.merge(
    duration.rename(columns={"mom_side_label": "mom_side_for_duration"}),
    on=["commodity", "mom_side_for_duration"],
    how="left",
).drop(columns=["mom_side_for_duration"])

q2_current["life_estimate_reliability"] = q2_current["duration_reliability"]

q3_current = _select(latest, [
    "commodity", "commodity_name", "main_symbol", "date", "close",
    "MOMMain", "MOMMainSide", "MOMMainExists", "MOMEstablished",
    "MOMValidatedEstablished", "MOMExtremeThr", "MOMExtreme", "MOMEndRisk",
    "MOMEnd_loss", "MOMEnd_flip", "MOMEnd_signal", "MOMEnd_path",
    "MOMEnd_volume", "MOMEndConfirmed", "MOMEnd",
    "MOMEpisodeID", "MOMEpisodeEndDate", "MOMEpisodeEndReason",
]).merge(_select(life, ["commodity", "mom_end_risk", "mom_end_confirmed"] + p_cols), on="commodity", how="left")

core = (
    q1_current
    .merge(_select(q2_current, [
        "commodity", "momentum_established_now", "mom_side_label", "mom_age",
        "mom_end_risk", "mom_end_confirmed", "n_comparable_completed",
        "duration_reliability", "mean_remaining_days", "median_remaining_days",
        "life_estimate_reliability",
    ] + p_cols), on="commodity", how="left")
    .merge(_select(q3_current, [
        "commodity", "MOMExtreme", "MOMEndRisk", "MOMEndConfirmed", "MOMEnd",
    ]), on="commodity", how="left", suffixes=("", "_end"))
)
core["momentum_status"] = core.apply(_mom_status, axis=1)
front = ["commodity", "commodity_name", "main_symbol", "date", "close", "momentum_status", "life_estimate_reliability"]
core = core[[c for c in front if c in core.columns] + [c for c in core.columns if c not in front]]

_state = panel_state_duration.copy()
_state["mom_exists_flag"] = _state["MOMMainExists"].fillna(0).astype(bool)
_state["mom_signal_established_flag"] = _state["MOMEstablished"].fillna(0).astype(bool)
_state["mom_established_flag"] = _state["MOMValidatedEstablished"].fillna(0).astype(bool)
_state["mom_up_flag"] = _state["mom_established_flag"] & (_state["MOMMainSide"] > 0)
_state["mom_down_flag"] = _state["mom_established_flag"] & (_state["MOMMainSide"] < 0)

q1_history = (
    _select(_state, ["commodity", "commodity_name", "main_symbol"])
    .drop_duplicates("commodity")
    .merge(_state.groupby("commodity").agg(
        first_date=("date", "min"),
        last_date=("date", "max"),
        total_obs=("date", "size"),
        mom_exists_days=("mom_exists_flag", "sum"),
        mom_signal_established_days=("mom_signal_established_flag", "sum"),
        mom_established_days=("mom_established_flag", "sum"),
        up_established_days=("mom_up_flag", "sum"),
        down_established_days=("mom_down_flag", "sum"),
    ).reset_index(), on="commodity", how="left")
    .merge(best, on="commodity", how="left")
)
q1_history["mom_exists_rate"] = q1_history["mom_exists_days"] / q1_history["total_obs"]
q1_history["mom_signal_established_rate"] = q1_history["mom_signal_established_days"] / q1_history["total_obs"]
q1_history["mom_established_rate"] = q1_history["mom_established_days"] / q1_history["total_obs"]
q1_history["up_share_when_established"] = q1_history["up_established_days"] / q1_history["mom_established_days"].replace(0, np.nan)
q1_history["down_share_when_established"] = q1_history["down_established_days"] / q1_history["mom_established_days"].replace(0, np.nan)

q2_history = duration.copy()
q3_history = momentum_survival_table.copy()
if {"commodity", "mom_side_label"}.issubset(q3_history.columns) and not duration.empty:
    q3_history = q3_history.merge(
        _select(duration, [
            "commodity", "mom_side_label", "hist_n_episodes", "hist_n_ended",
            "hist_end_risk_rate", "hist_duration_reliability",
        ]),
        on=["commodity", "mom_side_label"],
        how="left",
    )
if "n_state_obs" in q3_history.columns:
    q3_history["survival_sample_reliability"] = q3_history["n_state_obs"].apply(_reliability)

outputs = {
    "momentum_core_current_summary.csv": core,
    "momentum_q1_existence_current.csv": q1_current,
    "momentum_q2_duration_current.csv": q2_current,
    "momentum_q3_end_current.csv": q3_current,
    "momentum_hist_q1_existence_model.csv": q1_history,
    "momentum_hist_q2_duration_distribution.csv": q2_history,
    "momentum_hist_q3_end_risk_survival.csv": q3_history,
}

for filename, df_out in outputs.items():
    df_out.to_csv(RESULT_DIR / filename, index=False)

manifest = pd.DataFrame([
    {"file": "momentum_core_current_summary.csv", "scope": "CURRENT", "question": "ALL", "description": "当前动量主表：动量存在、持续周期、消失风险、最佳动量模型汇总"},
    {"file": "momentum_q1_existence_current.csv", "scope": "CURRENT", "question": "Q1", "description": "当前是否存在/确立动量：方向、强度、一致性、动量年龄"},
    {"file": "momentum_q2_duration_current.csv", "scope": "CURRENT", "question": "Q2", "description": "当前动量还能持续多久：剩余寿命、未来N日消失概率、历史同方向duration摘要"},
    {"file": "momentum_q3_end_current.csv", "scope": "CURRENT", "question": "Q3", "description": "当前动量是否消失/是否有消失风险：确认消失、极端延展、消失触发项"},
    {"file": "momentum_hist_q1_existence_model.csv", "scope": "FULL_HISTORY", "question": "Q1", "description": "全历史动量存在频率、方向分布、最佳动量模型有效性"},
    {"file": "momentum_hist_q2_duration_distribution.csv", "scope": "FULL_HISTORY", "question": "Q2", "description": "全历史动量episode持续时间分布：均值、中位数、分位数、样本数"},
    {"file": "momentum_hist_q3_end_risk_survival.csv", "scope": "FULL_HISTORY", "question": "Q3", "description": "全历史动量消失风险：按方向、risk状态、horizon统计未来消失概率"},
])
manifest.to_csv(RESULT_DIR / "momentum_output_manifest.csv", index=False)

display(core)
display(manifest)
print("Saved current + full-history momentum CSVs to:", RESULT_DIR.resolve())

,commodity,commodity_name,main_symbol,date,close,momentum_status,life_estimate_reliability,J_mom_star,K_mom_eval,mom_model_scope,MOMModelUsable,MOMMain,MOMMainSide,MOMMainMCS,MOMPathConfirm,MOMVolumeConfirm,MOMMainExists,MOMEstablished,MOMValidatedEstablished,MOMEpisodeID,MOMEpisodeSideLabel,MOMAge,MOMEpisodeStartDate,MOMEpisodeRawStartDate,best_model_label,best_J,best_K,best_n,best_mean_y,best_t_nw,best_hit_rate,best_boot_p_mean_le_0,best_mean_raw,best_candidate_model,best_confirmed_model,best_model_quality,momentum_established_now,mom_side_label,mom_age,mom_end_risk,mom_end_confirmed,n_comparable_completed,duration_reliability,mean_remaining_days,median_remaining_days,p_end_within_1d,p_end_within_2d,p_end_within_3d,p_end_within_5d,p_end_within_10d,p_end_within_20d,MOMExtreme,MOMEndRisk,MOMEndConfirmed,MOMEnd
0,AG,沪银,KQ.m@SHFE.ag,2026-06-30,13966.0,NO_ESTABLISHED_MOMENTUM,NO_ACTIVE_MOMENTUM,2.0,1.0,CONFIRMED,True,0.898357,1.0,1.000000,1.0,0.0,0.0,0.0,0.0,<NA>,<NA>,<NA>,<NA>,<NA>,AG_Momentum_J2_K1,2,1,800,0.106057,2.437516,0.518750,0.0025,0.000735,True,True,CONFIRMED,False,UP,<NA>,False,False,0,NO_ACTIVE_MOMENTUM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0
1,CF,棉花,KQ.m@CZCE.CF,2026-06-30,16065.0,NO_ESTABLISHED_MOMENTUM,NO_ACTIVE_MOMENTUM,3.0,2.0,CONFIRMED,True,0.778458,1.0,0.666667,1.0,1.0,0.0,0.0,0.0,<NA>,<NA>,<NA>,<NA>,<NA>,CF_Momentum_J3_K2,3,2,1209,0.138725,3.200463,0.510339,0.0000,0.001790,True,True,CONFIRMED,False,UP,<NA>,False,False,0,NO_ACTIVE_MOMENTUM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0
2,EB,苯乙烯,KQ.m@DCE.eb,2026-06-30,7368.0,DOWN_MOMENTUM_WITH_END_RISK,OK,20.0,5.0,CONFIRMED,True,-10.223056,-1.0,0.700000,1.0,0.0,1.0,1.0,1.0,EB_MOM_0069,DOWN,10,2026-06-16 00:00:00,2026-06-15 00:00:00,EB_Momentum_J20_K5,20,5,722,0.225588,2.361195,0.540166,0.0025,0.007204,True,True,CONFIRMED,True,DOWN,10,True,False,12,OK,8.750000,5.5,0.083333,0.083333,0.250000,0.500000,0.75,0.833333,1.0,1.0,0.0,0.0
3,FG,玻璃,KQ.m@CZCE.FG,2026-06-30,967.0,MOMENTUM_CONFIRMED_ENDED,NO_ACTIVE_MOMENTUM,40.0,1.0,CONFIRMED,True,-4.626710,-1.0,0.575000,0.0,0.0,0.0,0.0,0.0,<NA>,<NA>,<NA>,<NA>,<NA>,FG_Momentum_J40_K1,40,1,510,0.114226,2.129536,0.537255,0.0175,0.001370,True,True,CONFIRMED,False,DOWN,<NA>,False,True,0,NO_ACTIVE_MOMENTUM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,1.0,1.0
4,LH,生猪,KQ.m@DCE.lh,2026-06-30,12365.0,NO_ESTABLISHED_MOMENTUM,NO_ACTIVE_MOMENTUM,20.0,5.0,CANDIDATE,True,3.774473,1.0,0.500000,0.0,1.0,0.0,0.0,0.0,<NA>,<NA>,<NA>,<NA>,<NA>,LH_Momentum_J20_K5,20,5,569,0.232740,1.558231,0.548330,0.0575,0.005783,True,False,CANDIDATE,False,UP,<NA>,False,False,0,NO_ACTIVE_MOMENTUM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0
5,M,豆粕,KQ.m@DCE.m,2026-06-30,2951.0,UP_MOMENTUM_ESTABLISHED,OK,3.0,20.0,CANDIDATE,True,1.007103,1.0,0.666667,1.0,0.0,1.0,1.0,1.0,M_MOM_0309,UP,2,2026-06-29 00:00:00,2026-06-26 00:00:00,M_Momentum_J3_K20,3,20,1225,0.062819,1.360382,0.515918,0.0675,0.003155,True,False,CANDIDATE,True,UP,2,False,False,108,OK,2.037037,1.0,0.564815,0.731481,0.842593,0.944444,1.00,1.000000,0.0,0.0,0.0,0.0
6,MA,甲醇,KQ.m@CZCE.MA,2026-06-30,2442.0,NO_ESTABLISHED_MOMENTUM,NO_ACTIVE_MOMENTUM,3.0,20.0,DIAGNOSTIC_ONLY,False,0.012438,1.0,0.666667,1.0,0.0,0.0,0.0,0.0,<NA>,<NA>,<NA>,<NA>,<NA>,MA_Momentum_J3_K20,3,20,1292,0.033127,0.732274,0.503870,0.3250,0.002264,False,False,DIAGNOSTIC_ONLY,False,UP,<NA>,False,False,0,NO_ACTIVE_MOMENTUM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0
7,SA,纯碱,KQ.m@CZCE.SA,2026-06-30,1084.0,MOMENTUM_CONFIRMED_ENDED,NO_ACTIVE_MOMENTUM,2.0,2.0,CONFIRMED,True,-0.778070,-1.0,0.500000,0.0,0.0,0.0,0.0,0.0,<NA>,<NA>,<NA>,<NA>,<NA>,SA_Momentum_J2_K2,2,2,524,0.155320,2.357919,0.549618,0.0150,0.003686,True,True,CONFIRMED,False,DOWN,<NA>,False,True,0,NO_ACTIVE_MOMENTUM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,1.0,1.0
8,SP,纸浆,KQ.m@SHFE.sp,2026-06-30,4700.0,NO_ESTABLISHED_MOMENTUM,NO_ACTIVE_MOMENTUM,40.0,20.0,CONFIRMED,True,-10.586142,-1.0,0.575000,0.0,1.0,0.0,0.0,0.0,<NA>,<NA>,<NA>,<NA>,<NA>,SP_Momentum_J40_K20,40,20,357,0.577768,3.4072

,file,scope,question,description
0,momentum_core_current_summary.csv,CURRENT,ALL,当前动量主表：动量存在、持续周期、消失风险、最佳动量模型汇总
1,momentum_q1_existence_current.csv,CURRENT,Q1,当前是否存在/确立动量：方向、强度、一致性、动量年龄
2,momentum_q2_duration_current.csv,CURRENT,Q2,当前动量还能持续多久：剩余寿命、未来N日消失概率、历史同方向duration摘要
3,momentum_q3_end_current.csv,CURRENT,Q3,当前动量是否消失/是否有消失风险：确认消失、极端延展、消失触发项
4,momentum_hist_q1_existence_model.csv,FULL_HISTORY,Q1,全历史动量存在频率、方向分布、最佳动量模型有效性
5,momentum_hist_q2_duration_distribution.csv,FULL_HISTORY,Q2,全历史动量episode持续时间分布：均值、中位数、分位数、样本数
6,momentum_hist_q3_end_risk_survival.csv,FULL_HISTORY,Q3,全历史动量消失风险：按方向、risk状态、horizon统计未来消失概率


Saved current + full-history momentum CSVs to: /home/zilinm2/main_continuous_daily_trend_momentum_reversal_research/results
